# 🐳 Docker and CI/CD — Ship Your Model Service Reliably

> **What you'll learn:** what containers are (and aren't), how images, layers, and the build cache work, how to write a small, secure, **multi-stage Dockerfile** for an ML service, how to run, inspect, and stop containers, how to keep secrets out of images, how to wire a **GitHub Actions** pipeline that lints, tests, checks **model quality**, and publishes images, and how to roll out (and roll back) a new model safely. Every Docker command runs for real from this notebook, and every size and timing is measured.

| | |
|---|---|
| **Difficulty** | 🟢 Beginner → 🟡 Intermediate (🔴 quality gates and deployment strategies) |
| **Time** | ~4 hours to read and run, +2 hours for exercises and the project |
| **Prerequisites** | [FastAPI](01_FastAPI.ipynb) (the model service we containerize) · [Virtual Environments & Packaging](../00_Foundations/04_Virtual_Env_and_Packaging.ipynb) (requirements, lock files, uv) |
| **Tested with** | Docker Engine 28 (Docker Desktop, BuildKit, Compose v2) · Python 3.12 · uv 0.10 · fastapi 0.141 · scikit-learn 1.9 · pytest 9 · PyYAML 6 |
| **Interview relevance** | ⭐⭐⭐ High — layer caching, multi-stage builds, non-root containers, image size, containers vs VMs, CI vs CD, blue/green vs canary, rolling back a bad model, testing ML systems, where models and secrets live |

## 🤔 What Is Docker (and CI/CD)?

Your model service from the last notebook runs on *your* laptop, with *your* Python, *your* library versions, and *your* model file. A teammate's laptop, a cloud server, or a Kubernetes cluster has none of that. "It works on my machine" is not a deployment plan.

**Docker** packages the service *together with everything it needs* into one **image**: a slice of a Linux file system with Python, the exact libraries, your code, and the model. Running an image gives you a **container**, an isolated process that behaves the same on any machine with Docker.

Think of shipping containers. Before them, every cargo (barrels, sacks, cars) needed special handling at every port. A standard steel box made ships, cranes, and trucks interchangeable. Docker images are that standard box for software.

```
 Dockerfile  ──docker build──▶  image (read-only layers)  ──docker run──▶  container (a running process)
 (recipe)                       (the packaged box)                         (the box, opened and running)
                                       │
                                 docker push ──▶ registry (Docker Hub, GHCR, ECR…) ──▶ docker pull on any server
```

**CI/CD** automates everything around it:
- **CI — Continuous Integration:** every push runs lint, tests, and quality checks automatically, so broken code (or a worse model) is caught in minutes, not in production.
- **CD — Continuous Delivery / Deployment:** every change that passes CI is packaged (an image pushed to a registry) and is ready to deploy (*delivery*), or is deployed automatically (*deployment*).

**GitHub Actions** is GitHub's built-in CI/CD system: YAML files in `.github/workflows/` describe jobs that run on GitHub's machines.

## 🎯 Why It Matters

- **Containers are how models ship.** Kubernetes, AWS ECS/SageMaker, Google Cloud Run, Azure Container Apps, BentoML, Ray Serve, and KServe all run container images. An AI engineer is expected to write a sensible Dockerfile.
- **Size and startup time cost money.** A 2 GB image pulls slowly, so autoscaling reacts late. A root-user image with secrets baked in fails security review. You'll measure both.
- **ML needs extra CI gates.** Normal software is right if its tests pass. A model can pass every unit test and still be *worse* than yesterday's. CI for ML adds data validation and metric thresholds.
- **Deployments fail; rollbacks must be boring.** Canary releases and one-command rollbacks are standard interview material for ML platform and MLOps roles.
- **In interviews:** *"How does Docker layer caching work, and how do you order a Dockerfile?"*, *"Why multi-stage?"*, *"Why not run as root?"*, *"How do you shrink a 3 GB image?"*, *"Containers vs VMs?"*, *"CI vs CD?"*, *"Blue/green vs canary?"*, *"A new model is hurting conversions — what do you do?"*, *"How do you test an ML system?"*, *"Where do models and secrets live?"*

## ✅ By the End You Can

- [ ] Explain containers vs VMs, images vs containers, and exactly when the Docker build cache is reused
- [ ] Write a multi-stage Dockerfile for an ML service: slim base, pinned dependencies, `.dockerignore`, non-root user, `HEALTHCHECK`, env-var config
- [ ] Build, run, inspect, and stop containers, and measure image size and cold-start time
- [ ] Keep secrets out of images and choose between baking a model in or mounting it
- [ ] Write and validate a GitHub Actions workflow that lints, tests, gates on model quality, and publishes images on tags
- [ ] Compare rolling, blue/green, canary, and shadow deployments, and plan a rollback

## 📋 Table of Contents

1. [Containers vs Virtual Machines](#1.-Containers-vs-Virtual-Machines-🟢)
2. [Images, Layers, and the Build Cache](#2.-Images,-Layers,-and-the-Build-Cache-🟢)
3. [Your First Dockerfile for an ML Service](#3.-Your-First-Dockerfile-for-an-ML-Service-🟢)
4. [The Build Context and .dockerignore](#4.-The-Build-Context-and-.dockerignore-🟢)
5. [Running Containers: Ports, Env Vars, Logs, Cleanup](#5.-Running-Containers:-Ports,-Env-Vars,-Logs,-Cleanup-🟢)
6. [Multi-Stage Builds and Image Size](#6.-Multi-Stage-Builds-and-Image-Size-🟡)
7. [Security: Non-Root Users and Secrets](#7.-Security:-Non-Root-Users-and-Secrets-🟡)
8. [Health Checks, Configuration, and Model Artifacts](#8.-Health-Checks,-Configuration,-and-Model-Artifacts-🟡)
9. [Multi-Service Setups with Docker Compose](#9.-Multi-Service-Setups-with-Docker-Compose-🟡)
10. [CPU vs GPU Images and Image Scanning](#10.-CPU-vs-GPU-Images-and-Image-Scanning-🟢)
11. [CI with GitHub Actions](#11.-CI-with-GitHub-Actions-🟡)
12. [Testing ML Systems: The Testing Pyramid](#12.-Testing-ML-Systems:-The-Testing-Pyramid-🟡)
13. [ML Quality Gates in CI](#13.-ML-Quality-Gates-in-CI-🔴)
14. [Deployment Strategies and Rollback](#14.-Deployment-Strategies-and-Rollback-🔴)
- [🔧 Build It From Scratch](#🔧-Build-It-From-Scratch) · [⚠️ Common Pitfalls](#⚠️-Common-Pitfalls) · [🏋️ Practice Exercises](#🏋️-Practice-Exercises) · [🚀 Mini Project](#🚀-Mini-Project:-Release-Pipeline-for-the-Housing-Model-Service) · [🎤 Interview Q&A](#🎤-Interview-Q&A) · [🧪 Quick Quiz](#🧪-Quick-Quiz) · [📚 Resources](#📚-Resources) · [📝 Summary](#📝-Summary-Cheat-Sheet)

## ⚙️ Setup

**You need Docker running.** On macOS or Windows install [Docker Desktop](https://docs.docker.com/desktop/) and start it. On Linux install Docker Engine and start the service. The setup cell waits up to ~2 minutes for the Docker daemon (the background service that builds and runs containers) to answer. If it never does, every Docker cell prints a clear **⏭️ Skipped** message instead of results, and the Python-only parts (CI validation, quality gates, deployment analysis, exercises) still run.

**Downloads (first run only):** the `python:3.12-slim` base image (~45 MB compressed), the full `python:3.12` image for the "naive" comparison (~400 MB compressed), a small `uv` image, and the service's Python wheels inside the builds (~100 MB). Later runs reuse Docker's cache.

Everything the notebook writes goes to `_outputs/docker_nb/`. Every container it starts carries the label `mlcourse.notebook=docker-ci` and is removed at the end of each section.

The `run()` helper prints `$ command` and its output like a terminal. `check()` gives exercise feedback: **✅** correct, **⏳** not attempted, **❌** wrong (with a hint).

In [ ]:
# %pip install -q "fastapi>=0.135" "uvicorn>=0.35" "httpx>=0.28" "scikit-learn>=1.8" pyyaml pytest

import hashlib
import importlib.metadata as md
import json
import math
import os
import platform
import re
import secrets
import shutil
import socket
import stat
import subprocess
import sys
import textwrap
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

import httpx
import joblib
import matplotlib.pyplot as plt
import numpy as np
import yaml

for pkg in ["fastapi", "uvicorn", "scikit-learn", "pytest", "pyyaml", "httpx"]:
    print(f"{pkg} {md.version(pkg)}", end=" | ")
print(f"Python {sys.version.split()[0]} | host: {platform.system()} {platform.machine()}")

SEED = 42
OUTPUT_DIR = Path("_outputs").resolve()
WORK = OUTPUT_DIR / "docker_nb"
WORK.mkdir(parents=True, exist_ok=True)
RUN_ID = uuid.uuid4().hex[:8]                   # makes cache experiments independent of earlier notebook runs
LABEL = "mlcourse.notebook=docker-ci"           # attached to every container we start → easy cleanup
BASE_IMAGE = "python:3.12-slim"
UV_IMAGE = "ghcr.io/astral-sh/uv:0.10.3"        # the official uv image, pinned to an exact version
DOCKER = shutil.which("docker")
UV = shutil.which("uv")


def run(cmd, cwd=None, env=None, show=True, max_lines=25, tail=True, timeout=1200, input_text=None):
    """Run a command like a terminal: print `$ command`, its (trimmed) output, and the exit code if non-zero."""
    cmd = [str(c) for c in cmd]
    start = time.perf_counter()
    result = subprocess.run(cmd, cwd=cwd, env={**os.environ, **(env or {})}, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, timeout=timeout, input=input_text)       # one stream, so lines stay in order
    result.stdout = re.sub(r"\x1b\[[0-9;?]*[A-Za-z]", "", result.stdout)        # drop terminal colour/cursor codes
    result.stderr = ""
    result.seconds = time.perf_counter() - start
    if show:
        shown = ["docker" if c == DOCKER else "python" if c == sys.executable else "uv" if c == UV else c for c in cmd]
        print("$", " ".join(shown).replace(str(OUTPUT_DIR), "_outputs"))
        lines = result.stdout.rstrip().splitlines()
        hidden = len(lines) - max_lines
        if hidden > 0:
            lines = lines[-max_lines:] if tail else lines[:max_lines]
            print(f"  … ({hidden} {'earlier' if tail else 'more'} lines hidden)")
        for line in lines:
            print("  " + line.replace(str(OUTPUT_DIR), "_outputs"))
        if result.returncode != 0:
            print(f"  [exit code {result.returncode}]")
    return result


def docker_daemon_status(wait_seconds=120):
    """(True, info) when the Docker daemon answers; waits while Docker Desktop is still starting."""
    if DOCKER is None:
        return False, "the `docker` command is not installed"
    deadline = time.monotonic() + wait_seconds
    while True:
        try:
            probe = subprocess.run([DOCKER, "info", "--format", "{{json .}}"], capture_output=True, text=True, timeout=60)
            info = json.loads(probe.stdout) if probe.returncode == 0 and probe.stdout.strip() else {}
            if probe.returncode == 0 and info.get("ServerVersion"):
                return True, info
            reason = (probe.stderr.strip() or "no response").splitlines()[-1]
        except (subprocess.TimeoutExpired, json.JSONDecodeError) as err:
            reason = str(err)
        if time.monotonic() > deadline:
            return False, reason
        print("  waiting for the Docker daemon …")
        time.sleep(5)


DOCKER_OK, DOCKER_INFO = docker_daemon_status()
if DOCKER_OK:
    print(f"Docker Engine {DOCKER_INFO['ServerVersion']} | {DOCKER_INFO['OperatingSystem']} | {DOCKER_INFO['OSType']}/{DOCKER_INFO['Architecture']} | "
          f"{DOCKER_INFO['NCPU']} CPUs | {DOCKER_INFO['MemTotal'] / 1e9:.1f} GB RAM for containers")
    stale = subprocess.run([DOCKER, "ps", "-aq", "--filter", f"label={LABEL}"], capture_output=True, text=True).stdout.split()
    if stale:
        subprocess.run([DOCKER, "rm", "-f", *stale], capture_output=True)
        print(f"removed {len(stale)} leftover container(s) from an earlier run")
    if subprocess.run([DOCKER, "image", "inspect", BASE_IMAGE], capture_output=True).returncode != 0:
        run([DOCKER, "pull", BASE_IMAGE], max_lines=3)
else:
    print(f"⚠️ Docker is not available: {DOCKER_INFO}. Docker cells will print skip messages.")
print("uv:", UV or "not installed (the ruff lint step will be skipped)")


def docker_skip(what):
    print(f"⏭️ Skipped: {what} needs a running Docker daemon ({DOCKER_INFO if not DOCKER_OK else ''}). "
          "Start Docker Desktop (or the Docker service on Linux) and re-run this cell.")


def check(name, got, expected, hint=""):
    """✅ if correct, ⏳ if not attempted yet (None), ❌ AssertionError with a hint otherwise."""
    if got is None or got is ...:
        print(f"⏳ {name}: not attempted yet — replace None with your answer.")
        return

    def close(a, b):
        if isinstance(b, float) or isinstance(a, float):
            return isinstance(a, (int, float)) and isinstance(b, (int, float)) and math.isclose(a, b, rel_tol=1e-6, abs_tol=1e-9)
        if isinstance(b, dict):
            return isinstance(a, dict) and a.keys() == b.keys() and all(close(a[k], b[k]) for k in b)
        if isinstance(b, (list, tuple)):
            return isinstance(a, (list, tuple)) and len(a) == len(b) and all(close(x, y) for x, y in zip(a, b))
        return a == b

    assert close(got, expected), f"❌ {name}: not quite (got {got!r}). {hint}"
    print(f"✅ {name}: correct!")


def write(path, text, mode=None):
    """Create a file (and its folders) from an indented triple-quoted string."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(text).lstrip("\n"))
    if mode is not None:
        path.chmod(mode)


def tree(root, skip=(".pytest_cache", "__pycache__", ".ruff_cache")):
    root = Path(root)
    print(f"📁 {root.name}/")
    for p in sorted(root.rglob("*")):
        parts = p.relative_to(root).parts
        if any(part in skip for part in parts):
            continue
        size = f"  ({p.stat().st_size / 1e6:.1f} MB)" if p.is_file() and p.stat().st_size > 1e6 else ""
        print("    " * len(parts) + ("📁 " if p.is_dir() else "📄 ") + p.name + ("/" if p.is_dir() else "") + size)


STEP_RE = re.compile(r"^#(\d+) \[(?:([\w.-]+) )?(\d+)/(\d+)\] (.+)$")


def parse_build_steps(log):
    """Parse `docker build --progress=plain` output → one dict per Dockerfile step, with cached True/False."""
    steps, cached_ids = {}, set()
    for line in log.splitlines():
        m = STEP_RE.match(line)
        if m and m.group(1) not in steps:
            steps[m.group(1)] = {"stage": m.group(2) or "", "step": int(m.group(3)), "instruction": m.group(5)}
        elif re.match(r"^#(\d+) CACHED$", line):
            cached_ids.add(line.split()[0][1:])
    ordered = sorted(steps.items(), key=lambda kv: (kv[1]["stage"], kv[1]["step"]))
    return [{**info, "cached": step_id in cached_ids} for step_id, info in ordered]


def docker_build(context, tag, dockerfile=None, target=None, extra=(), show_steps=True):
    """docker build with plain progress output; prints each step as CACHED or BUILT and the total time."""
    cmd = [DOCKER, "build", "--progress=plain", "-t", tag]
    if dockerfile:
        cmd += ["-f", str(dockerfile)]
    if target:
        cmd += ["--target", target]
    cmd += [*extra, str(context)]
    result = run(cmd, show=False)
    log = result.stdout
    if result.returncode != 0:
        print("\n".join(log.splitlines()[-40:]))
        raise RuntimeError(f"docker build failed for {tag}")
    result.steps = parse_build_steps(log)
    context_sizes = re.findall(r"transferring context: ([\d.]+)([kMG]?B)", log)
    result.context_bytes = to_bytes(*context_sizes[-1]) if context_sizes else 0
    if show_steps:
        print(f"built {tag} in {result.seconds:.1f} s (build context {result.context_bytes / 1e6:.2f} MB)")
        for s in result.steps:
            if s["instruction"].startswith("FROM"):
                continue
            print(f"   {'CACHED' if s['cached'] else 'BUILT ':6}  [{s['stage'] + ' ' if s['stage'] else ''}{s['step']}] {s['instruction'][:95]}")
    return result


def to_bytes(number, unit):
    """Docker prints decimal units: 1 kB = 1000 B, 1 MB = 1000² B, 1 GB = 1000³ B."""
    return float(number) * {"B": 1, "kB": 1e3, "MB": 1e6, "GB": 1e9}[unit]


def image_size(tag):
    """(disk size as `docker image ls` reports it, compressed content size) in bytes."""
    listed = json.loads(subprocess.run([DOCKER, "image", "ls", "--format", "{{json .}}", tag], capture_output=True, text=True).stdout.splitlines()[0])
    number, unit = re.match(r"([\d.]+)([kMG]?B)", listed["Size"]).groups()
    compressed = int(subprocess.run([DOCKER, "image", "inspect", "--format", "{{.Size}}", tag], capture_output=True, text=True).stdout.strip())
    return to_bytes(number, unit), compressed


def free_port():
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


def docker_run_service(image, name, env=None, volumes=(), extra=(), container_port=8000):
    """Start a detached, labelled container with a random host port. Returns the host port."""
    host_port = free_port()
    cmd = [DOCKER, "run", "-d", "--name", name, "--label", LABEL, "-p", f"127.0.0.1:{host_port}:{container_port}"]
    for key, value in (env or {}).items():
        cmd += ["-e", f"{key}={value}"]
    for volume in volumes:
        cmd += ["-v", volume]
    result = run([*cmd, *extra, image], show=False)
    if result.returncode != 0:
        raise RuntimeError(result.stdout)
    return host_port


def wait_until_ready(url, container=None, timeout=90):
    """Poll `url` until it returns 200. Returns seconds waited; shows container logs on failure."""
    start = time.perf_counter()
    while time.perf_counter() - start < timeout:
        try:
            if httpx.get(url, timeout=2).status_code == 200:
                return time.perf_counter() - start
        except httpx.TransportError:
            pass
        if container and subprocess.run([DOCKER, "inspect", "-f", "{{.State.Running}}", container], capture_output=True, text=True).stdout.strip() == "false":
            break
        time.sleep(0.1)
    logs = subprocess.run([DOCKER, "logs", container], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) if container else None
    raise RuntimeError(f"{url} not ready after {time.perf_counter() - start:.0f} s" + (f"\n{logs.stdout[-2500:]}" if logs else ""))


def remove_container(name):
    subprocess.run([DOCKER, "rm", "-f", name], capture_output=True)

## 1. Containers vs Virtual Machines 🟢

Both give you an isolated place to run software. They isolate at different levels:

```
       VIRTUAL MACHINES                               CONTAINERS
 ┌─────────┐ ┌─────────┐ ┌─────────┐          ┌─────────┐ ┌─────────┐ ┌─────────┐
 │  app A  │ │  app B  │ │  app C  │          │  app A  │ │  app B  │ │  app C  │
 │  libs   │ │  libs   │ │  libs   │          │  libs   │ │  libs   │ │  libs   │
 │ guest OS│ │ guest OS│ │ guest OS│          └─────────┘ └─────────┘ └─────────┘
 │ + kernel│ │ + kernel│ │ + kernel│          ───── container runtime (Docker) ─────
 └─────────┘ └─────────┘ └─────────┘          ────────── ONE shared host kernel ──────────
 ─────────── hypervisor ────────────
 ─────────── hardware ──────────────                  hardware
```

- A **virtual machine (VM)** emulates a whole computer: its own operating-system **kernel** (the core of the OS that talks to hardware). Strong isolation, but gigabytes of disk and boot times measured in tens of seconds.
- A **container** is an ordinary process on the host that the Linux kernel *fences off*: **namespaces** give it its own view of files, processes, and network; **cgroups** limit its CPU and memory. No second kernel, so it starts in milliseconds and adds little overhead.

| | Virtual machine | Container |
|---|---|---|
| Kernel | its own | shares the host's |
| Start time | tens of seconds to minutes | milliseconds to seconds |
| Size | GBs (full OS) | MBs (just app + libraries) |
| Isolation | strong (hardware-level) | good, weaker (shared kernel) |
| Can run a different OS | yes (Windows on Linux) | only the host's kernel family (Linux containers need a Linux kernel) |

**On a Mac or Windows laptop** there is no Linux kernel, so Docker Desktop runs *one* small Linux VM and all containers share *its* kernel. Let's see that for real.

In [ ]:
print(f"Host (this notebook): {platform.system()} {platform.release()} on {platform.machine()}")
if not DOCKER_OK:
    docker_skip("running a container")
else:
    run([DOCKER, "run", "--rm", BASE_IMAGE, "python", "-c",
         "import platform, sys; print('Container:', platform.system(), platform.release(), 'on', platform.machine(), '| Python', sys.version.split()[0])"])
    run([DOCKER, "run", "--rm", BASE_IMAGE, "sh", "-c", "head -2 /etc/os-release; echo processes visible inside: $(ls /proc | grep -c '^[0-9]')"])

    start_times = []
    for _ in range(5):
        start = time.perf_counter()
        subprocess.run([DOCKER, "run", "--rm", BASE_IMAGE, "true"], check=True, capture_output=True)
        start_times.append(time.perf_counter() - start)
    print(f"\ncreate + start + run + remove a container: median {np.median(start_times) * 1000:.0f} ms (5 runs)")

    probe = run([DOCKER, "run", "--rm", BASE_IMAGE, "python", "-c", "import sklearn"], show=False)
    print(f"scikit-learn on the host: {md.version('scikit-learn')} | inside a fresh container: "
          f"{'not installed — the container only has what its image contains' if probe.returncode else 'installed'}")

The container reports **Linux** even though this notebook runs on your host OS (on a Mac: a Linux kernel from Docker Desktop's VM), sees only its own handful of processes, and has none of your host's Python packages. It is isolated, and it starts in well under a second.

### ✍️ Your Turn

For each statement, answer `"vm"`, `"container"`, or `"both"`.

In [ ]:
which_one = {
    "runs its own operating-system kernel": None,
    "typically starts in well under a second": None,
    "isolates its processes and files from other workloads": None,
    "can run a Windows guest on a Linux host": None,
    "is distributed as layered, content-addressed images": None,
}
check("containers_vs_vms", None if None in which_one.values() else which_one,
      {"runs its own operating-system kernel": "vm", "typically starts in well under a second": "container",
       "isolates its processes and files from other workloads": "both", "can run a Windows guest on a Linux host": "vm",
       "is distributed as layered, content-addressed images": "container"},
      hint="Containers share the host kernel; VMs virtualize hardware and boot a full OS.")

<details><summary>💡 Show solution</summary>

```python
which_one = {
    "runs its own operating-system kernel": "vm",
    "typically starts in well under a second": "container",
    "isolates its processes and files from other workloads": "both",
    "can run a Windows guest on a Linux host": "vm",
    "is distributed as layered, content-addressed images": "container",
}
check("containers_vs_vms", which_one,
      {"runs its own operating-system kernel": "vm", "typically starts in well under a second": "container",
       "isolates its processes and files from other workloads": "both", "can run a Windows guest on a Linux host": "vm",
       "is distributed as layered, content-addressed images": "container"})
```

VM images exist too (AMIs, qcow2 files), but they are whole-disk snapshots, not stacks of shared, content-addressed layers.
</details>

> 💡 **Interview angle:** "Containers vs VMs?" — containers share the host kernel and isolate with namespaces and cgroups: fast, small, dense. VMs virtualize hardware with their own kernel: stronger isolation, heavier. In practice they're combined: cloud Kubernetes nodes are VMs running many containers, and sandboxes like gVisor or Firecracker narrow the gap for untrusted workloads.

## 2. Images, Layers, and the Build Cache 🟢

An **image** is a stack of read-only **layers**. Each file-changing instruction in a Dockerfile (`RUN`, `COPY`, `ADD`) produces one layer: a diff of files added or changed. A container adds a thin writable layer on top. Layers are identified by the hash of their content, so two images based on `python:3.12-slim` store those base layers only once.

```
 container   ┌──────────────────────────────┐  ← writable, thrown away when the container is removed
             ├──────────────────────────────┤
 your image  │ COPY app/ ./app/             │  ← small, changes often
             │ RUN pip install -r reqs.txt  │  ← big, changes rarely
             │ COPY requirements.txt .      │
             ├──────────────────────────────┤
 base image  │ python:3.12-slim layers      │  ← shared by every image built on it
             └──────────────────────────────┘
```

**The build cache rule** (the #1 Docker interview question). For each step, Docker reuses the cached layer if:
1. the **parent** layer is the same, **and**
2. the **instruction text** is the same, **and**
3. for `COPY`/`ADD`, the **copied files' contents** (and permissions) are the same. Modification times don't count.

The moment one step misses the cache, **every step after it is rebuilt**. So order instructions from *rarely changes* (base, system packages, dependencies) to *changes all the time* (your code).

In [ ]:
if not DOCKER_OK:
    docker_skip("inspecting image layers")
else:
    run([DOCKER, "history", "--format", "table {{.Size}}\t{{.CreatedBy}}", BASE_IMAGE], max_lines=14, tail=False)
    layers = json.loads(run([DOCKER, "image", "inspect", "--format", "{{json .RootFS.Layers}}", BASE_IMAGE], show=False).stdout)
    digest = run([DOCKER, "image", "inspect", "--format", "{{index .RepoDigests 0}}", BASE_IMAGE], show=False).stdout.strip()
    disk, compressed = image_size(BASE_IMAGE)
    print(f"\n{BASE_IMAGE}: {len(layers)} file-system layers | {compressed / 1e6:.0f} MB compressed download | {disk / 1e6:.0f} MB on disk")
    print("immutable content digest:", digest)

Only a few history entries have a non-zero size: `ENV`, `CMD`, `WORKDIR`, and `LABEL` change the image's *metadata*, not its files.

Now the cache rule in action. Two Dockerfiles install the same small package and copy the same app. The only difference is **order**. We build both, change one line of application code, and rebuild.

In [ ]:
CACHE_DEMO = WORK / "cache_order"
write(CACHE_DEMO / "requirements.txt", "six==1.17.0\n")
write(CACHE_DEMO / "app.py", f'RUN_ID = "{RUN_ID}"\nprint("model service v1")\n')
write(CACHE_DEMO / "Dockerfile.bad", f"""
    FROM {BASE_IMAGE}
    WORKDIR /srv
    # ❌ copies everything first: ANY file change invalidates the pip install below
    COPY . .
    RUN pip install --no-cache-dir -r requirements.txt
    CMD ["python", "app.py"]
""")
write(CACHE_DEMO / "Dockerfile.good", f"""
    FROM {BASE_IMAGE}
    WORKDIR /srv
    # ✅ dependencies first: this layer is reused until requirements.txt changes
    COPY requirements.txt .
    RUN pip install --no-cache-dir -r requirements.txt
    COPY app.py .
    CMD ["python", "app.py"]
""")

if not DOCKER_OK:
    docker_skip("the build-cache experiment")
else:
    print("── first builds ──")
    for variant in ["bad", "good"]:
        docker_build(CACHE_DEMO, f"cache-demo:{variant}", dockerfile=CACHE_DEMO / f"Dockerfile.{variant}")
    (CACHE_DEMO / "app.py").write_text(f'RUN_ID = "{RUN_ID}"\nprint("model service v2")   # one line of code changed\n')
    print("\n── after changing one line in app.py ──")
    rebuild = {variant: docker_build(CACHE_DEMO, f"cache-demo:{variant}", dockerfile=CACHE_DEMO / f"Dockerfile.{variant}") for variant in ["bad", "good"]}
    pip_rebuilt = {v: any("pip install" in s["instruction"] and not s["cached"] for s in r.steps) for v, r in rebuild.items()}
    print(f"\npip install re-ran → bad order: {pip_rebuilt['bad']}, good order: {pip_rebuilt['good']} | "
          f"rebuild time: bad {rebuild['bad'].seconds:.1f} s vs good {rebuild['good'].seconds:.1f} s")
    print("container says:", run([DOCKER, "run", "--rm", "cache-demo:good"], show=False).stdout.strip())

With a real ML image, that `pip install` is scikit-learn, PyTorch, or transformers: minutes and gigabytes instead of seconds. Good ordering turns every code-change rebuild into a few seconds.

### ✍️ Your Turn

Put these Dockerfile lines in the most **cache-friendly** valid order.

In [ ]:
scrambled = [
    'CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]',
    "COPY app/ ./app/",
    "RUN pip install --no-cache-dir -r requirements.txt",
    "FROM python:3.12-slim",
    "COPY requirements.txt .",
    "WORKDIR /srv",
]
cache_friendly = None  # TODO: a list with the same six lines in the best order
check("cache_friendly_order", cache_friendly,
      ["FROM python:3.12-slim", "WORKDIR /srv", "COPY requirements.txt .", "RUN pip install --no-cache-dir -r requirements.txt",
       "COPY app/ ./app/", 'CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]'],
      hint="Base → working dir → dependency list → install → code → start command.")

<details><summary>💡 Show solution</summary>

```python
cache_friendly = [
    "FROM python:3.12-slim",
    "WORKDIR /srv",
    "COPY requirements.txt .",
    "RUN pip install --no-cache-dir -r requirements.txt",
    "COPY app/ ./app/",
    'CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]',
]
check("cache_friendly_order", cache_friendly,
      ["FROM python:3.12-slim", "WORKDIR /srv", "COPY requirements.txt .", "RUN pip install --no-cache-dir -r requirements.txt",
       "COPY app/ ./app/", 'CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]'])
```

`CMD` only sets metadata, so its position barely affects caching, but by convention it goes last.
</details>

> 💡 **Interview angle:** "How does layer caching work and how do you order a Dockerfile?" — each step's cache key is the parent layer + the instruction + (for `COPY`/`ADD`) a checksum of the copied files; the first miss rebuilds everything after it. Copy the dependency manifest and install **before** copying code. In CI, reuse the cache across runs (`--cache-from`, BuildKit cache mounts, or GitHub Actions cache).

## 3. Your First Dockerfile for an ML Service 🟢

The instructions you'll use 95% of the time:

| Instruction | What it does | Tip |
|---|---|---|
| `FROM image:tag` | start from a base image | pin a specific tag (`python:3.12-slim`), never `latest` |
| `WORKDIR /srv` | set (and create) the working directory | avoids `cd` in every command |
| `COPY src dest` | copy files from the build context | copy dependency files before code |
| `RUN command` | run a command at **build** time, creating a layer | chain related commands with `&&` in one `RUN` |
| `ENV KEY=value` | environment variable at build and run time | good for defaults, **never** for secrets |
| `ARG NAME` | variable available only at build time | also visible in image history, so no secrets either |
| `EXPOSE 8000` | documents the port the app listens on | doesn't publish it; `docker run -p` does |
| `USER name` | run later steps and the container as this user | don't run as root (Section 7) |
| `HEALTHCHECK CMD …` | command Docker runs to judge health | Section 8 |
| `CMD ["exe", "arg"]` | default command when the container starts | use the JSON **exec form** so signals reach your app |

First, the service to ship: the California Housing price model from the FastAPI notebook, condensed to the essentials. We train the model here so this notebook stands on its own.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

housing = fetch_california_housing(as_frame=True)
FEATURES = list(housing.feature_names)
X_all, y_all = housing.data.to_numpy(), housing.target.to_numpy()          # target: median house value in $100,000s
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=SEED)

model_v1 = HistGradientBoostingRegressor(max_leaf_nodes=31, max_iter=300, random_state=SEED).fit(X_train, y_train)
test_mae = mean_absolute_error(y_test, model_v1.predict(X_test))
baseline_mae = mean_absolute_error(y_test, np.full_like(y_test, y_train.mean()))
print(f"model v1: test MAE {test_mae:.3f} | R² {r2_score(y_test, model_v1.predict(X_test)):.3f} | mean-baseline MAE {baseline_mae:.3f}")

SERVICE = WORK / "housing_service"


def save_model(model, model_dir, version, X_fit, metrics):
    model_dir = Path(model_dir)
    model_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, model_dir / "model.joblib")
    (model_dir / "metadata.json").write_text(json.dumps({
        "model_version": version, "features": FEATURES, "sklearn_version": md.version("scikit-learn"),
        "trained_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "metrics": metrics,
        "training_ranges": {name: [float(X_fit[:, i].min()), float(X_fit[:, i].max())] for i, name in enumerate(FEATURES)},
    }, indent=2))


save_model(model_v1, SERVICE / "model", "housing-hgb-1.0.0", X_train,
           {"test_mae": round(test_mae, 4), "baseline_test_mae": round(baseline_mae, 4)})

write(SERVICE / "app" / "__init__.py", "")
write(SERVICE / "app" / "schemas.py", '''
    from pydantic import BaseModel, ConfigDict, Field

    MAX_BATCH_SIZE = 256


    class HouseFeatures(BaseModel):
        model_config = ConfigDict(extra="forbid")
        MedInc: float = Field(gt=0, le=25, allow_inf_nan=False)
        HouseAge: float = Field(ge=0, le=100, allow_inf_nan=False)
        AveRooms: float = Field(gt=0, le=200, allow_inf_nan=False)
        AveBedrms: float = Field(gt=0, le=50, allow_inf_nan=False)
        Population: float = Field(ge=1, le=100_000, allow_inf_nan=False)
        AveOccup: float = Field(gt=0, le=2000, allow_inf_nan=False)
        Latitude: float = Field(ge=32, le=42.5, allow_inf_nan=False)
        Longitude: float = Field(ge=-125, le=-114, allow_inf_nan=False)


    class BatchRequest(BaseModel):
        model_config = ConfigDict(extra="forbid")
        rows: list[HouseFeatures] = Field(min_length=1, max_length=MAX_BATCH_SIZE)
''')
write(SERVICE / "app" / "model.py", '''
    import json
    from dataclasses import dataclass
    from pathlib import Path

    import joblib
    import numpy as np


    @dataclass
    class ModelService:
        model: object
        metadata: dict

        @classmethod
        def load(cls, model_dir) -> "ModelService":
            model_dir = Path(model_dir)
            return cls(joblib.load(model_dir / "model.joblib"), json.loads((model_dir / "metadata.json").read_text()))

        @property
        def version(self) -> str:
            return self.metadata["model_version"]

        def predict(self, rows: list[dict]) -> list[dict]:
            features = self.metadata["features"]
            ranges = self.metadata["training_ranges"]
            values = self.model.predict(np.array([[row[f] for f in features] for row in rows], dtype=float))
            return [{"median_house_value": round(float(v), 4),
                     "out_of_range_features": [f for f in features if not ranges[f][0] <= row[f] <= ranges[f][1]]}
                    for row, v in zip(rows, values)]
''')
write(SERVICE / "app" / "main.py", '''
    """Housing price service: config from env vars, model loaded once, API-key auth, 422 for bad input."""
    import logging
    import os
    import secrets
    import time
    from contextlib import asynccontextmanager
    from typing import Annotated

    from fastapi import Depends, FastAPI, HTTPException, Request, Security
    from fastapi.exceptions import RequestValidationError
    from fastapi.responses import JSONResponse
    from fastapi.security import APIKeyHeader

    from app.model import ModelService
    from app.schemas import BatchRequest, HouseFeatures

    logger = logging.getLogger("uvicorn.error")


    @asynccontextmanager
    async def lifespan(app: FastAPI):
        start = time.perf_counter()
        app.state.model = ModelService.load(os.environ.get("MODEL_DIR", "model"))
        app.state.api_keys = [k for k in os.environ.get("API_KEYS", "").split(",") if k]
        logger.info("loaded model %s in %.0f ms", app.state.model.version, (time.perf_counter() - start) * 1000)
        yield
        app.state.model = None
        logger.info("model unloaded")


    app = FastAPI(title="Housing Price Service", version="1.0.0", lifespan=lifespan)


    @app.exception_handler(RequestValidationError)
    async def validation_handler(request: Request, exc: RequestValidationError):
        return JSONResponse(status_code=422, content={"detail": [{"loc": list(e["loc"]), "msg": e["msg"]} for e in exc.errors()]})


    def require_api_key(request: Request, api_key: Annotated[str | None, Security(APIKeyHeader(name="X-API-Key", auto_error=False))]):
        if api_key is None or not any(secrets.compare_digest(api_key.encode(), k.encode()) for k in request.app.state.api_keys):
            raise HTTPException(status_code=401, detail="Invalid or missing API key")


    def get_model(request: Request) -> ModelService:
        if getattr(request.app.state, "model", None) is None:
            raise HTTPException(status_code=503, detail="Model not loaded")
        return request.app.state.model


    @app.get("/health")
    def health():
        return {"status": "ok"}


    @app.get("/ready")
    def ready(model: Annotated[ModelService, Depends(get_model)]):
        return {"status": "ready", "model_version": model.version}


    @app.post("/v1/predict", dependencies=[Depends(require_api_key)])
    def predict(features: HouseFeatures, model: Annotated[ModelService, Depends(get_model)]):
        return {**model.predict([features.model_dump()])[0], "model_version": model.version}


    @app.post("/v1/predict/batch", dependencies=[Depends(require_api_key)])
    def predict_batch(batch: BatchRequest, model: Annotated[ModelService, Depends(get_model)]):
        predictions = model.predict([row.model_dump() for row in batch.rows])
        return {"model_version": model.version, "n": len(predictions), "predictions": predictions}
''')

runtime_packages = ["fastapi", "starlette", "pydantic", "uvicorn", "scikit-learn", "numpy", "scipy", "joblib"]
(SERVICE / "requirements.txt").write_text("".join(f"{p}=={md.version(p)}\n" for p in runtime_packages))
# starlette 1.6's TestClient uses the `httpx2` package (plain `httpx` still works but raises a deprecation warning)
(SERVICE / "requirements-dev.txt").write_text("-r requirements.txt\n" + "".join(f"{p}=={md.version(p)}\n" for p in ["pytest", "httpx2"]))
print((SERVICE / "requirements.txt").read_text())

Now the Dockerfile. It's single-stage and deliberately plain. Later sections improve it step by step.

In [ ]:
write(SERVICE / "Dockerfile.simple", f"""
    FROM {BASE_IMAGE}

    # Don't write .pyc files at runtime; flush logs immediately
    ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1
    WORKDIR /srv

    # 1) dependencies — cached until requirements.txt changes
    COPY requirements.txt .
    RUN pip install --no-cache-dir -r requirements.txt

    # 2) code and model — change often, so they come last
    COPY app/ ./app/
    COPY model/ ./model/

    ENV MODEL_DIR=/srv/model
    EXPOSE 8000
    # exec form (JSON list): uvicorn runs as PID 1 and receives stop signals directly
    CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
""")

if not DOCKER_OK:
    docker_skip("building the service image")
else:
    simple_build = docker_build(SERVICE, "housing-api:simple", dockerfile=SERVICE / "Dockerfile.simple")
    disk, compressed = image_size("housing-api:simple")
    print(f"housing-api:simple → {disk / 1e6:.0f} MB on disk, {compressed / 1e6:.0f} MB compressed")
    run([DOCKER, "run", "--rm", "housing-api:simple", "python", "-c",
         "import sklearn, fastapi; print('inside the image: scikit-learn', sklearn.__version__, '| fastapi', fastapi.__version__)"])

> 💡 **Interview angle:** "Walk me through a Dockerfile for a model API." — slim, pinned base; dependency manifest copied and installed first (no pip cache); code and model copied after; config via `ENV` defaults overridden at run time; `EXPOSE` the port; `CMD` in exec form binding `0.0.0.0`. Then improve it: `.dockerignore`, multi-stage, non-root user, `HEALTHCHECK`.

## 4. The Build Context and .dockerignore 🟢

`docker build <folder>` first sends that whole folder, the **build context**, to the Docker daemon. Anything in it can end up in the image through `COPY . .`: raw datasets, notebooks, `.git`, virtual environments, and **`.env` files with real secrets**.

A **`.dockerignore`** file (same syntax idea as `.gitignore`) removes paths from the context before they're sent. Smaller context → faster builds, smaller images, no accidental leaks.

To see the damage, we add what real project folders contain: a 50 MB data dump (random bytes, standing in for raw training data), a notebook, and a `.env` file with a fake secret. Then we build with the lazy `COPY . .`.

In [ ]:
(SERVICE / "data").mkdir(exist_ok=True)
(SERVICE / "data" / "raw_training_dump.bin").write_bytes(np.random.default_rng(SEED).bytes(50_000_000))
write(SERVICE / "notebooks" / "exploration.ipynb", '{"cells": [], "metadata": {}, "nbformat": 4, "nbformat_minor": 5}\n')
write(SERVICE / ".env", "API_KEYS=sk-live-THIS-KEY-MUST-NEVER-SHIP\n")
write(SERVICE / "Dockerfile.copyall", f"""
    FROM {BASE_IMAGE}
    ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1
    WORKDIR /srv
    COPY requirements.txt .
    RUN pip install --no-cache-dir -r requirements.txt
    # ❌ copies the entire build context
    COPY . .
    ENV MODEL_DIR=/srv/model
    CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
""")
(SERVICE / ".dockerignore").unlink(missing_ok=True)

if not DOCKER_OK:
    docker_skip("the build-context experiment")
else:
    no_ignore = docker_build(SERVICE, "housing-api:copyall", dockerfile=SERVICE / "Dockerfile.copyall")
    leaked = run([DOCKER, "run", "--rm", "housing-api:copyall", "sh", "-c", "ls -a /srv && echo '--- /srv/.env:' && cat /srv/.env"])
    size_leaky, _ = image_size("housing-api:copyall")

In [ ]:
write(SERVICE / ".dockerignore", """
    # Never send these to the Docker daemon
    .git
    .env
    *.env
    .venv/
    __pycache__/
    *.pyc
    .pytest_cache/
    .ruff_cache/
    data/
    notebooks/
    *.ipynb
""")

if not DOCKER_OK:
    docker_skip("the .dockerignore rebuild")
else:
    with_ignore = docker_build(SERVICE, "housing-api:copyall", dockerfile=SERVICE / "Dockerfile.copyall")
    listing = run([DOCKER, "run", "--rm", "housing-api:copyall", "ls", "-a", "/srv"])
    size_clean, _ = image_size("housing-api:copyall")
    assert ".env" not in listing.stdout.split(), ".env must not be in the image"
    print(f"\nbuild context: {no_ignore.context_bytes / 1e6:.1f} MB → {with_ignore.context_bytes / 1e6:.2f} MB | "
          f"image: {size_leaky / 1e6:.0f} MB → {size_clean / 1e6:.0f} MB | secret in image: yes → no")

# remove the stand-in junk so later steps (CI, tests) work on a clean project folder
shutil.rmtree(SERVICE / "data", ignore_errors=True)
shutil.rmtree(SERVICE / "notebooks", ignore_errors=True)
(SERVICE / ".env").unlink(missing_ok=True)

Even with a `.dockerignore`, prefer copying **explicit paths** (`COPY app/ ./app/`) over `COPY . .`. The ignore file is a safety net, not the plan.

> 💡 **Interview angle:** "Our image is 1 GB bigger than expected and builds are slow." — check the build context size (BuildKit prints it), add a `.dockerignore` for data, `.git`, venvs, and notebooks, and copy explicit paths. Then check layers with `docker history` and apply multi-stage builds.

## 5. Running Containers: Ports, Env Vars, Logs, Cleanup 🟢

| Command | What it does |
|---|---|
| `docker run -d --name api -p 8080:8000 -e API_KEYS=… image` | start in the background (`-d`), publish container port 8000 on host port 8080, set an env var |
| `docker ps` / `docker ps -a` | list running / all containers |
| `docker logs -f api` | read (follow) the container's stdout/stderr |
| `docker exec -it api sh` | run a command inside a running container (debugging) |
| `docker stats --no-stream` | live CPU and memory usage |
| `docker stop -t 10 api` | send **SIGTERM**, wait up to 10 s for a graceful exit, then SIGKILL (pass `-t` explicitly: the default can be changed by the daemon's configuration) |
| `docker rm api` / `docker run --rm …` | delete a stopped container / delete automatically on exit |
| `docker image ls` / `docker system df` / `docker image prune` | list images / disk usage / clean up dangling images |

**Port mapping:** inside the container the app listens on port 8000. That port is private until you publish it with `-p HOST:CONTAINER`. We bind to `127.0.0.1` on a random free host port so nothing is exposed to the network.

In [ ]:
API_KEY = secrets.token_urlsafe(16)                  # generated at runtime, passed in with -e (never baked into the image)
sample_row = dict(zip(FEATURES, map(float, X_test[0])))

if not DOCKER_OK:
    docker_skip("running the service container")
else:
    name = f"housing-simple-{RUN_ID}"
    host_port = docker_run_service("housing-api:simple", name, env={"API_KEYS": API_KEY})
    base_url = f"http://127.0.0.1:{host_port}"
    ready_seconds = wait_until_ready(f"{base_url}/ready", container=name)
    print(f"container ready after {ready_seconds:.2f} s at {base_url}")
    run([DOCKER, "ps", "--filter", f"name={name}", "--format", "table {{.Names}}\t{{.Image}}\t{{.Status}}\t{{.Ports}}"])

    response = httpx.post(f"{base_url}/v1/predict", json=sample_row, headers={"X-API-Key": API_KEY})
    print("\nPOST /v1/predict →", response.status_code, response.json())
    print(f"offline model says {model_v1.predict(X_test[:1])[0]:.4f} | without a key → {httpx.post(f'{base_url}/v1/predict', json=sample_row).status_code}")

    run([DOCKER, "exec", name, "sh", "-c", "echo MODEL_DIR=$MODEL_DIR; whoami; python --version"])
    run([DOCKER, "stats", "--no-stream", "--format", "table {{.Name}}\t{{.CPUPerc}}\t{{.MemUsage}}", name])
    run([DOCKER, "logs", name], max_lines=12)

    stop = run([DOCKER, "stop", "-t", "10", name], show=False)
    exit_code = int(run([DOCKER, "inspect", "-f", "{{.State.ExitCode}}", name], show=False).stdout.strip())
    meaning = {0: "clean exit", 143: "ended by SIGTERM (128 + 15) after shutting down", 137: "killed with SIGKILL (128 + 9) — it did not stop in time"}
    print(f"\ndocker stop took {stop.seconds:.2f} s → exit code {exit_code}: {meaning.get(exit_code, 'see docs')}")
    remove_container(name)

The logs show the whole lifecycle: startup, `loaded model`, the requests, then `Shutting down` and `model unloaded` after `docker stop`. Exit codes above 128 mean "ended by signal number (code − 128)". Orchestrators treat **137** (killed) as a problem, and ⚠️ Pitfall 1 shows how to cause it by accident.

### ✍️ Your Turn

Write `signal_for_exit_code(code)` that returns the **name of the signal** that ended a container (`"SIGKILL"`, `"SIGTERM"`, …) when `code > 128`, and `None` for normal exit codes (0 = success, 1–128 = the program itself chose that code).

In [ ]:
def signal_for_exit_code(code):
    return ...  # TODO (return None for normal exits)


exit_codes = [0, 1, 3, 137, 143, 130]
got = None if signal_for_exit_code(137) is ... else [signal_for_exit_code(c) for c in exit_codes]
check("signal_for_exit_code", got, [None, None, None, "SIGKILL", "SIGTERM", "SIGINT"],
      hint="import signal; signal.Signals(code - 128).name")

<details><summary>💡 Show solution</summary>

```python
import signal

def signal_for_exit_code(code):
    return signal.Signals(code - 128).name if code > 128 else None

exit_codes = [0, 1, 3, 137, 143, 130]
check("signal_for_exit_code", [signal_for_exit_code(c) for c in exit_codes], [None, None, None, "SIGKILL", "SIGTERM", "SIGINT"])
```

130 = 128 + 2 is SIGINT, what you get after pressing Ctrl+C on `docker run` in a terminal. An `OOMKilled` container also exits with 137: check `docker inspect -f '{{.State.OOMKilled}}'` to tell the two apart.
</details>

> 💡 **Interview angle:** "The container runs but I can't reach the API." — check that the port is published (`-p`), that the app binds `0.0.0.0` (not `127.0.0.1`) inside the container, `docker ps` for status and ports, `docker logs` for startup errors, and `docker exec` to curl from inside.

## 6. Multi-Stage Builds and Image Size 🟡

Image size matters: every new server or autoscaled replica must **pull** the image before it can start, registries charge for storage and transfer, and every extra package is more attack surface.

The biggest wins, in order:
1. **A slim base image.** `python:3.12` (full Debian with compilers and headers) vs `python:3.12-slim`.
2. **Only runtime dependencies.** No pytest, linters, or notebooks in the shipped image.
3. **No package caches.** `pip install --no-cache-dir`, or uv with a BuildKit cache mount that stays outside the image.
4. **Multi-stage builds.** Do the messy work (tools, compilers, downloading wheels) in a **builder** stage, then `COPY --from=builder` only the finished result into a clean **runtime** stage. Nothing else from the builder ships.

```
 stage "builder" (python:3.12-slim + uv)            stage "runtime" (python:3.12-slim)
 ┌─────────────────────────────────────┐            ┌─────────────────────────────────┐
 │ uv binary, caches, build tools      │            │ /opt/venv   ← COPY --from=builder│
 │ /opt/venv with installed packages ──┼───────────▶│ app/, model/                    │
 └─────────────────────────────────────┘ discarded  │ non-root user, HEALTHCHECK, CMD │
                                                    └─────────────────────────────────┘
```

Let's build a typical first attempt ("naive") and an optimized multi-stage image of the same service, then measure.

In [ ]:
write(SERVICE / "Dockerfile.naive", """
    # A very common first Dockerfile: full base image, everything copied, dev tools and pip cache included
    FROM python:3.12
    WORKDIR /srv
    COPY . .
    RUN pip install -r requirements-dev.txt
    ENV MODEL_DIR=/srv/model
    CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
""")

write(SERVICE / "Dockerfile", f"""
    # syntax=docker/dockerfile:1

    # ---------- stage 1: builder — has uv and caches, never shipped ----------
    FROM {BASE_IMAGE} AS builder
    COPY --from={UV_IMAGE} /uv /bin/uv
    ENV UV_COMPILE_BYTECODE=1 UV_LINK_MODE=copy UV_PYTHON_DOWNLOADS=never
    COPY requirements.txt /tmp/requirements.txt
    RUN --mount=type=cache,target=/root/.cache/uv \\
        uv venv /opt/venv && \\
        uv pip install --python /opt/venv/bin/python -r /tmp/requirements.txt

    # ---------- stage 2: runtime — just Python, the venv, code, and model ----------
    FROM {BASE_IMAGE} AS runtime
    # OMP_NUM_THREADS=1: one OpenMP thread per prediction; scikit-learn's default (all cores per call) oversubscribes the CPU under concurrent requests
    ENV PYTHONDONTWRITEBYTECODE=1 \\
        PYTHONUNBUFFERED=1 \\
        PATH="/opt/venv/bin:$PATH" \\
        MODEL_DIR=/srv/model \\
        OMP_NUM_THREADS=1
    RUN groupadd --system --gid 10001 app && useradd --system --uid 10001 --gid app --no-create-home app
    WORKDIR /srv
    COPY --from=builder /opt/venv /opt/venv
    COPY app/ ./app/
    COPY model/ ./model/
    USER app
    EXPOSE 8000
    HEALTHCHECK --interval=30s --timeout=3s --start-period=30s --start-interval=1s --retries=3 \\
        CMD ["python", "-c", "import sys, urllib.request; sys.exit(urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2).status != 200)"]
    CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
""")

sizes = {}
if not DOCKER_OK:
    docker_skip("building and measuring the naive and optimized images")
else:
    naive_build = docker_build(SERVICE, "housing-api:naive", dockerfile=SERVICE / "Dockerfile.naive", show_steps=False)
    print(f"built housing-api:naive in {naive_build.seconds:.0f} s")
    optimized_build = docker_build(SERVICE, "housing-api:optimized", target="runtime")
    for tag in ["housing-api:naive", "housing-api:simple", "housing-api:optimized"]:
        sizes[tag] = image_size(tag)
    print(f"\n{'image':24} {'on disk':>9} {'compressed (pull size)':>24}")
    for tag, (disk, compressed) in sizes.items():
        print(f"{tag:24} {disk / 1e6:7.0f} MB {compressed / 1e6:21.0f} MB")

In [ ]:
if DOCKER_OK:
    naive_disk, naive_pull = sizes["housing-api:naive"]
    opt_disk, opt_pull = sizes["housing-api:optimized"]
    print(f"optimized vs naive: {1 - opt_disk / naive_disk:.0%} smaller on disk, {1 - opt_pull / naive_pull:.0%} smaller to pull "
          f"({naive_pull / opt_pull:.1f}× less data for every new server)")
    simple_pull = sizes["housing-api:simple"][1]
    gap = (simple_pull - opt_pull) / simple_pull
    print(f"optimized (multi-stage) vs simple (careful single-stage): {gap:+.1%} pull size → "
          + ("about the same: every dependency installed from a pre-built wheel, so there was nothing to leave behind in the builder"
             if abs(gap) < 0.05 else "the multi-stage build left build-time files behind"))

    fig, ax = plt.subplots(figsize=(7, 3))
    labels = [t.split(":")[1] for t in sizes]
    ax.barh(labels, [d / 1e6 for d, _ in sizes.values()], label="on disk", color="tab:blue")
    ax.barh(labels, [c / 1e6 for _, c in sizes.values()], label="compressed (pull)", color="tab:orange")
    ax.set(xlabel="MB", title="Same service, three Dockerfiles")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print("largest layers of the optimized image:")
    history = run([DOCKER, "history", "--format", "{{.Size}}\t{{.CreatedBy}}", "housing-api:optimized"], show=False).stdout.splitlines()
    for line in history:
        size_text, created_by = line.split("\t", 1)
        if not size_text.startswith("0"):
            print(f"  {size_text:>8}  {created_by[:100]}")
else:
    docker_skip("the image-size comparison")

Reading the numbers above:
- The **base image** is most of the naive image's size: full `python:3.12` ships compilers, headers, and hundreds of Debian packages the service never uses. The naive image also carries **pytest**, the **pip cache**, and every file in the context.
- The careful single-stage `simple` image and the multi-stage `optimized` image come out about the same size. That's an honest result: all our dependencies install from **pre-built wheels**, so the builder stage has nothing heavy to leave behind. Multi-stage builds pay off in size when the build needs things the runtime doesn't: compilers (`build-essential`), Rust or Node toolchains, CUDA `-devel` images, downloaded source archives. They also keep build tools (here `uv`) out of production, and give you a `test` stage (used in the mini project).
- The optimized image's biggest layer is `/opt/venv` (NumPy, SciPy, scikit-learn). Next steps would be removing unused packages, or exporting the model to ONNX and serving it with a smaller runtime.

Other options: `python:3.12-alpine` is smaller but uses musl libc, so many scientific wheels don't install and must compile from source. Distroless images (`gcr.io/distroless/python3`) drop the shell and package manager entirely: smaller and more secure, but harder to debug.

> 💡 **Interview angle:** "How would you shrink a 3 GB PyTorch serving image?" — slim base; CPU-only or exact-CUDA wheels; multi-stage so compilers and caches stay in the builder; only runtime deps; `.dockerignore`; model weights mounted or downloaded at startup instead of baked in (if pull time dominates); check `docker history` for the fattest layers; consider ONNX/TensorRT runtimes. Always measure before and after.

## 7. Security: Non-Root Users and Secrets 🟡

**Why not run as root?** By default a container process runs as **root (uid 0)**. It's isolated, but if an attacker exploits your app (a deserialization bug, a malicious model file, a dependency vulnerability), root inside the container can modify your code, install tools, and has a much better chance of escaping through a kernel or runtime bug. The optimized image creates a system user `app` (uid 10001) and switches to it with `USER app`. Many Kubernetes clusters refuse root containers outright (`runAsNonRoot`).

In [ ]:
if not DOCKER_OK:
    docker_skip("the non-root comparison")
else:
    for tag in ["housing-api:simple", "housing-api:optimized"]:
        identity = run([DOCKER, "run", "--rm", tag, "id"], show=False).stdout.strip()
        tamper = run([DOCKER, "run", "--rm", tag, "sh", "-c",
                      "{ echo 'import os  # injected' >> /srv/app/main.py; } 2>/dev/null && echo 'modified the app code!' || echo 'permission denied'"],
                     show=False).stdout.strip()
        print(f"{tag:24} {identity[:45]:45} → try to overwrite app code: {tamper}")

**Secrets** (API keys, database passwords, Hugging Face tokens) must never be part of an image. Images get pushed to registries, cached on CI machines, and shared, and **anyone who can pull an image can read everything in it, including its history.** Three classic leaks:

| ❌ Leak | Where it's visible |
|---|---|
| `ENV API_KEY=sk-…` | `docker image inspect` (image config) |
| `ARG HF_TOKEN` used in a `RUN` | `docker history` records the build argument |
| `COPY . .` with a `.env` file | the file itself (Section 4) |

The fixes: **runtime** secrets are injected when the container starts (`-e`, `--env-file`, Kubernetes Secrets, cloud secret managers). **Build-time** secrets (e.g. a token to download a private package) use BuildKit **secret mounts**: the file exists only while that one `RUN` executes.

In [ ]:
SECRET_DEMO = WORK / "secrets_demo"
FAKE_TOKEN = "hf_demo_TOKEN_" + RUN_ID
write(SECRET_DEMO / "Dockerfile.leaky", f"""
    FROM {BASE_IMAGE}
    ARG HF_TOKEN
    ENV API_KEYS=sk-live-baked-into-the-image
    RUN test -n "$HF_TOKEN" && echo "downloaded private weights" > /download.log
    CMD ["cat", "/download.log"]
""")
write(SECRET_DEMO / "Dockerfile.safe", f"""
    # syntax=docker/dockerfile:1
    FROM {BASE_IMAGE}
    RUN --mount=type=secret,id=hf_token \\
        test -s /run/secrets/hf_token && echo "downloaded private weights" > /download.log
    CMD ["cat", "/download.log"]
""")
token_file = SECRET_DEMO / "hf_token.txt"
token_file.write_text(FAKE_TOKEN)

if not DOCKER_OK:
    docker_skip("the secrets experiment")
else:
    docker_build(SECRET_DEMO, "secrets-demo:leaky", dockerfile=SECRET_DEMO / "Dockerfile.leaky", extra=["--build-arg", f"HF_TOKEN={FAKE_TOKEN}"], show_steps=False)
    docker_build(SECRET_DEMO, "secrets-demo:safe", dockerfile=SECRET_DEMO / "Dockerfile.safe", extra=["--secret", f"id=hf_token,src={token_file}"], show_steps=False)
    for tag in ["secrets-demo:leaky", "secrets-demo:safe"]:
        config = run([DOCKER, "image", "inspect", tag], show=False).stdout
        history = run([DOCKER, "history", "--no-trunc", tag], show=False).stdout
        works = run([DOCKER, "run", "--rm", tag], show=False).stdout.strip()
        print(f"{tag:20} build worked: {works!r:30} | token in history: {FAKE_TOKEN in history} | "
              f"API key in image config: {'sk-live-baked-into-the-image' in config}")

Both builds used the token successfully, but only the safe one left no trace of it in the image.

**Where secrets live:** in a secret manager (AWS Secrets Manager, GCP Secret Manager, Vault, Kubernetes Secrets), injected at run time; in CI, as encrypted repository secrets, or better, **OIDC federation** so the pipeline gets short-lived cloud credentials without storing any key.

### ✍️ Your Turn

Write `find_baked_secrets(dockerfile_text)` that returns the **1-based line numbers** of `ENV` or `ARG` lines that assign a **non-empty value** to a name containing `KEY`, `TOKEN`, `SECRET`, or `PASSWORD` (case-insensitive).

In [ ]:
dockerfile_to_scan = """FROM python:3.12-slim
ARG PIP_INDEX_URL
ENV MODEL_DIR=/srv/model
ENV OPENAI_API_KEY=sk-proj-abc123
ARG HF_TOKEN=hf_live_987
RUN --mount=type=secret,id=hf_token cat /run/secrets/hf_token > /dev/null
env db_password="hunter2"
ARG GITHUB_TOKEN
CMD ["uvicorn", "app.main:app"]"""


def find_baked_secrets(text):
    return None  # TODO


check("find_baked_secrets", find_baked_secrets(dockerfile_to_scan), [4, 5, 7],
      hint=r"re.match(r'\s*(ENV|ARG)\s+(\w*(KEY|TOKEN|SECRET|PASSWORD)\w*)\s*=\s*\S+', line, re.IGNORECASE)")

<details><summary>💡 Show solution</summary>

```python
dockerfile_to_scan = """FROM python:3.12-slim
ARG PIP_INDEX_URL
ENV MODEL_DIR=/srv/model
ENV OPENAI_API_KEY=sk-proj-abc123
ARG HF_TOKEN=hf_live_987
RUN --mount=type=secret,id=hf_token cat /run/secrets/hf_token > /dev/null
env db_password="hunter2"
ARG GITHUB_TOKEN
CMD ["uvicorn", "app.main:app"]"""

SECRET_LINE = re.compile(r"\s*(ENV|ARG)\s+(\w*(KEY|TOKEN|SECRET|PASSWORD)\w*)\s*=\s*\S+", re.IGNORECASE)

def find_baked_secrets(text):
    return [number for number, line in enumerate(text.splitlines(), start=1) if SECRET_LINE.match(line)]

check("find_baked_secrets", find_baked_secrets(dockerfile_to_scan), [4, 5, 7])
```

`ARG GITHUB_TOKEN` (line 8) has no default, but passing `--build-arg GITHUB_TOKEN=…` and using it in a `RUN` still records it in history, as the experiment above showed. Real scanners (gitleaks, trufflehog, Trivy's secret scanner) look at files, image layers, and git history.
</details>

> 💡 **Interview angle:** "Why non-root, and where do secrets go?" — non-root limits what an exploited process can do and is required by many clusters; add a read-only root filesystem and dropped capabilities for defence in depth. Secrets are injected at run time from a secret manager, never `ENV`/`ARG`/`COPY`; build-time secrets use BuildKit secret mounts; CI uses OIDC instead of long-lived keys.

## 8. Health Checks, Configuration, and Model Artifacts 🟡

**`HEALTHCHECK`** tells Docker how to test the container. Its status (`starting` → `healthy` / `unhealthy`) appears in `docker ps` and is used by Docker Compose (`depends_on: condition: service_healthy`) and Docker Swarm. Kubernetes ignores it and uses its own liveness/readiness probes, but the idea is the same. Our optimized image checks `/health` every 30 s, and every 1 s during a 30 s start period.

**Configuration** comes from environment variables with safe defaults baked in (`ENV MODEL_DIR=/srv/model`, and `OMP_NUM_THREADS=1` so each concurrent prediction uses one CPU thread instead of all of them), overridden per environment at `docker run` time. One image goes to dev, staging, and prod unchanged: **build once, configure at deploy**.

**Cold start** is the time from "start a container" to "ready for traffic". It sets how fast autoscaling reacts. We measure it for the optimized image, including how long Docker takes to mark the container healthy.

In [ ]:
def cold_start(image, runs=3, env=None, volumes=()):
    """Median seconds from `docker run` to /ready == 200 (container start + imports + model load)."""
    times, health_times = [], []
    for i in range(runs):
        name = f"coldstart-{RUN_ID}-{i}-{image.replace(':', '-').replace('/', '-')}"
        start = time.perf_counter()
        port = docker_run_service(image, name, env={"API_KEYS": API_KEY, **(env or {})}, volumes=volumes)
        wait_until_ready(f"http://127.0.0.1:{port}/ready", container=name)
        times.append(time.perf_counter() - start)
        health = None
        while time.perf_counter() - start < 60:
            health = run([DOCKER, "inspect", "-f", "{{if .State.Health}}{{.State.Health.Status}}{{else}}none{{end}}", name], show=False).stdout.strip()
            if health in ("healthy", "none", "unhealthy"):
                break
            time.sleep(0.25)
        health_times.append((health, time.perf_counter() - start))
        remove_container(name)
    return float(np.median(times)), health_times


if not DOCKER_OK:
    docker_skip("measuring cold start and health status")
else:
    cold = {tag: cold_start(tag) for tag in ["housing-api:simple", "housing-api:optimized"]}
    for tag, (seconds, health) in cold.items():
        statuses = sorted({status for status, _ in health})
        print(f"{tag:24} cold start (docker run → /ready): {seconds:.2f} s | Docker health status: {statuses} after ~{np.median([t for _, t in health]):.1f} s")
    difference = cold["housing-api:simple"][0] - cold["housing-api:optimized"][0]
    print(f"→ difference {difference:+.2f} s: {'similar' if abs(difference) < 0.5 else 'noticeable'} — on one machine start time is dominated by "
          "Python imports and model loading. Image size matters most when a NEW node must pull the image first.")

**Where does the model file live?** Two common patterns:

| | **Baked into the image** (`COPY model/`) | **Mounted or downloaded at startup** (`-v`, init container, S3/registry download) |
|---|---|---|
| Reproducibility | one immutable artifact: image tag = code + model | must pin model version separately (registry alias, checksum) |
| Rollback | redeploy the previous image tag | point to the previous model version |
| Image size / pull | grows with the model (bad for multi-GB LLMs) | small image; model cached per node |
| Swap model without rebuild | ❌ | ✅ |
| Typical use | small/medium tabular and NLP models | large models, frequent retraining, many models per server |

The same optimized image can serve a **different model version** without rebuilding: mount a folder over `MODEL_DIR`.

In [ ]:
model_v2 = HistGradientBoostingRegressor(max_leaf_nodes=63, max_iter=400, learning_rate=0.08, random_state=SEED).fit(X_train, y_train)
v2_mae = mean_absolute_error(y_test, model_v2.predict(X_test))
MODEL_STORE = WORK / "model_store"
save_model(model_v2, MODEL_STORE / "housing-hgb-1.1.0", "housing-hgb-1.1.0", X_train, {"test_mae": round(v2_mae, 4), "baseline_test_mae": round(baseline_mae, 4)})
print(f"model v2 trained: test MAE {v2_mae:.4f} (v1: {test_mae:.4f})")

if not DOCKER_OK:
    docker_skip("serving a mounted model")
else:
    name = f"housing-mounted-{RUN_ID}"
    port = docker_run_service("housing-api:optimized", name, env={"API_KEYS": API_KEY, "MODEL_DIR": "/models/current"},
                              volumes=[f"{MODEL_STORE / 'housing-hgb-1.1.0'}:/models/current:ro"])
    wait_until_ready(f"http://127.0.0.1:{port}/ready", container=name)
    served = httpx.get(f"http://127.0.0.1:{port}/ready").json()["model_version"]
    prediction = httpx.post(f"http://127.0.0.1:{port}/v1/predict", json=sample_row, headers={"X-API-Key": API_KEY}).json()["median_house_value"]
    print(f"same image, mounted model → serving {served}; prediction {prediction} (v2 offline: {model_v2.predict(X_test[:1])[0]:.4f}, "
          f"v1 offline: {model_v1.predict(X_test[:1])[0]:.4f})")
    remove_container(name)

> 💡 **Interview angle:** "Where do you store models?" — in a **model registry** or object storage (MLflow Model Registry, S3/GCS with versioned paths, Hugging Face Hub, or baked into an immutable image for small models), always addressed by an **immutable version** (not `latest`), with metadata (training data version, metrics, library versions). The serving config pins the version, so rollback = change one pointer.

## 9. Multi-Service Setups with Docker Compose 🟡

Real ML systems are rarely one container: an API plus Redis (caching, rate limits), Postgres, a vector database, MLflow, a worker. **Docker Compose** describes several containers, their networks, environment, volumes, and start order in one `compose.yaml`, and starts them with one command. It's ideal for local development and CI integration tests. Production clusters use Kubernetes or a cloud service instead, with the same images.

Key ideas in the file below:
- Services reach each other by **service name** as a hostname (`http://api:8000`) on a private network Compose creates.
- `depends_on: condition: service_healthy` waits for the API image's **HEALTHCHECK** before starting the smoke test.
- `${API_KEYS:?…}` reads the key from *your* environment at run time. It's never written in the file.

In [ ]:
write(SERVICE / "smoke_test.py", '''
    """Runs in its own container: calls the API by its Compose service name; non-zero exit = failure."""
    import json
    import os
    import sys
    import urllib.request

    BASE = os.environ["API_URL"]
    KEY = os.environ["API_KEYS"].split(",")[0]
    ROW = {"MedInc": 8.3252, "HouseAge": 41.0, "AveRooms": 6.984, "AveBedrms": 1.024,
           "Population": 322.0, "AveOccup": 2.556, "Latitude": 37.88, "Longitude": -122.23}

    ready = json.load(urllib.request.urlopen(f"{BASE}/ready", timeout=5))
    request = urllib.request.Request(f"{BASE}/v1/predict", data=json.dumps(ROW).encode(),
                                     headers={"Content-Type": "application/json", "X-API-Key": KEY})
    prediction = json.load(urllib.request.urlopen(request, timeout=5))
    print("smoke test → ready:", ready, "| prediction:", prediction)
    sys.exit(0 if ready["status"] == "ready" and prediction["median_house_value"] > 0 else 1)
''')

write(SERVICE / "compose.yaml", f"""
    services:
      api:
        build:
          context: .
          target: runtime
        image: housing-api:optimized
        environment:
          API_KEYS: ${{API_KEYS:?set API_KEYS in your environment}}
        labels:
          mlcourse.notebook: docker-ci

      smoke-test:
        image: {BASE_IMAGE}
        depends_on:
          api:
            condition: service_healthy
        environment:
          API_URL: http://api:8000
          API_KEYS: ${{API_KEYS}}
        volumes:
          - ./smoke_test.py:/smoke_test.py:ro
        command: ["python", "/smoke_test.py"]
        labels:
          mlcourse.notebook: docker-ci
""")

if not DOCKER_OK:
    docker_skip("docker compose")
else:
    project = f"mlcourse-{RUN_ID}"
    compose_env = {"API_KEYS": API_KEY}
    up = run([DOCKER, "compose", "-p", project, "up", "--build", "--exit-code-from", "smoke-test"], cwd=SERVICE, env=compose_env, max_lines=12)
    down = run([DOCKER, "compose", "-p", project, "down", "--remove-orphans"], cwd=SERVICE, env=compose_env, max_lines=6)
    print(f"\ncompose up exit code = smoke-test exit code = {up.returncode} → {'integration test PASSED' if up.returncode == 0 else 'integration test FAILED'} "
          f"({up.seconds:.1f} s including health-check wait)")

> 💡 **Interview angle:** "How do you run integration tests that need the model API plus Redis/Postgres?" — Compose (or Testcontainers) in CI: start the stack, wait on health checks, run a test container against service names, use its exit code as the CI result, then `compose down`. The same images are promoted to production.

## 10. CPU vs GPU Images and Image Scanning 🟢

**GPU images.** To use an NVIDIA GPU, a container needs the CUDA **user-space libraries** inside the image and the **NVIDIA driver** on the host, connected by the NVIDIA Container Toolkit (`docker run --gpus all …`). The host driver must be new enough for the image's CUDA version. The CUDA libraries are huge, so sizes explode. Docker Hub's API reports the real compressed sizes:

In [ ]:
def hub_size(repository, tag):
    """Compressed linux/amd64 size of an image tag, from Docker Hub's public API."""
    data = httpx.get(f"https://hub.docker.com/v2/repositories/{repository}/tags/{tag}", timeout=20).json()
    return next(img["size"] for img in data["images"] if img.get("os") == "linux" and img.get("architecture") == "amd64")


def newest_tag(repository, pattern):
    """Most recently updated tag whose name matches a regex."""
    page = httpx.get(f"https://hub.docker.com/v2/repositories/{repository}/tags",
                     params={"page_size": 100, "ordering": "last_updated"}, timeout=20).json()
    return next(r["name"] for r in page["results"] if re.fullmatch(pattern, r["name"]))


try:
    torch_tag = newest_tag("pytorch/pytorch", r"\d+\.\d+\.\d+-cuda[\d.]+-cudnn\d+-runtime")
    cuda_tag = newest_tag("nvidia/cuda", r"\d+\.\d+\.\d+-runtime-ubuntu24\.04")
    candidates = {
        "python:3.12-slim (CPU)": ("library/python", "3.12-slim"),
        "python:3.12 (full, CPU)": ("library/python", "3.12"),
        f"nvidia/cuda:{cuda_tag}": ("nvidia/cuda", cuda_tag),
        f"pytorch/pytorch:{torch_tag}": ("pytorch/pytorch", torch_tag),
    }
    hub_sizes = {label: hub_size(*ref) for label, ref in candidates.items()}
    for label, size in hub_sizes.items():
        print(f"{label:48} {size / 1e9:6.2f} GB compressed (linux/amd64)")
    slim = hub_sizes["python:3.12-slim (CPU)"]
    print(f"\n→ the PyTorch CUDA runtime image is {hub_sizes[f'pytorch/pytorch:{torch_tag}'] / slim:.0f}× the slim Python base before your code is even added")
except (httpx.HTTPError, StopIteration, KeyError) as err:
    print(f"⏭️ Skipped: couldn't reach Docker Hub's API ({type(err).__name__}); needs network access.")

if shutil.which("nvidia-smi"):
    print("NVIDIA GPU detected — try: docker run --rm --gpus all nvidia/cuda:<version>-base-ubuntu24.04 nvidia-smi")
else:
    print(f"⏭️ No NVIDIA GPU on this {platform.system()} machine, so no GPU container is run here. "
          "On a Linux GPU host with the NVIDIA Container Toolkit: `docker run --rm --gpus all nvidia/cuda:<version>-base-ubuntu24.04 nvidia-smi`.")

Practical rules:
- **CPU-only serving?** Don't inherit a CUDA image. For PyTorch, install CPU wheels (`pip install torch --index-url https://download.pytorch.org/whl/cpu`) into a slim base.
- **GPU serving?** Use a `-runtime` CUDA base (not `-devel`, which adds compilers) and build in a `-devel` stage only if you compile extensions. Match the CUDA version to your framework wheels and the cluster's drivers.
- **Apple Silicon:** Docker on macOS runs Linux containers in a VM with **no GPU (Metal/MPS) passthrough**, so containerized PyTorch uses the CPU there.

**Image scanning.** Base images and wheels carry known vulnerabilities (CVEs). Scanners compare the packages in an image against vulnerability databases:

| Tool | Notes |
|---|---|
| **Trivy** (Aqua) | open source, scans images, filesystems, IaC, and secrets; common in CI (`aquasecurity/trivy-action`) |
| **Grype** (Anchore) | open source image/SBOM scanner; pairs with **Syft** to generate SBOMs |
| **Docker Scout** | built into Docker Desktop; needs a Docker account |
| Cloud registries | ECR, Artifact Registry, and ACR can scan on push |

The cheapest vulnerability reduction is shipping fewer packages. We can count what each image contains:

In [ ]:
if not DOCKER_OK:
    docker_skip("counting installed packages")
else:
    for tag in ["housing-api:naive", "housing-api:optimized"]:
        debs = run([DOCKER, "run", "--rm", "--entrypoint", "sh", tag, "-c", "dpkg -l | grep -c '^ii'"], show=False).stdout.strip()
        wheels = run([DOCKER, "run", "--rm", "--entrypoint", "python", tag, "-c",
                      "import importlib.metadata as m; print(len(list(m.distributions())))"], show=False).stdout.strip()
        print(f"{tag:24} Debian packages: {debs:>4} | Python distributions: {wheels:>3}")

scanners = {name: shutil.which(name) for name in ["trivy", "grype"]}
available = [name for name, path in scanners.items() if path]
if not DOCKER_OK:
    docker_skip("image scanning")
elif available:
    tool = available[0]
    command = [tool, "image", "--severity", "HIGH,CRITICAL", "--quiet", "housing-api:optimized"] if tool == "trivy" else [tool, "housing-api:optimized", "--only-fixed"]
    run(command, max_lines=30)
else:
    print("⏭️ Skipped the vulnerability scan: neither Trivy nor Grype is installed here. "
          "Install one (e.g. `brew install trivy`), then run `trivy image --severity HIGH,CRITICAL housing-api:optimized`.")

> 💡 **Interview angle:** "How do you keep images secure over time?" — minimal base and runtime-only packages; pin versions and **rebuild regularly** so base-image security fixes arrive; scan in CI and fail on fixable HIGH/CRITICAL CVEs; generate an SBOM; sign images (cosign) and deploy by digest; non-root user and read-only filesystem at run time.

## 11. CI with GitHub Actions 🟡

**CI vs CD, precisely:**
- **Continuous Integration** — every push and pull request is automatically linted, tested, and built. Goal: main is always in a working state.
- **Continuous Delivery** — every change that passes CI produces a deployable, versioned artifact (an image in a registry). A human presses "deploy".
- **Continuous Deployment** — the same, but deployment to production is automatic too.

**GitHub Actions vocabulary:** a *workflow* (YAML in `.github/workflows/`) is triggered by *events* (`on: push`, `pull_request`, tags); it has *jobs* that run on fresh virtual machines (`runs-on: ubuntu-latest`) in parallel unless linked with `needs`; each job has *steps* that either `run:` a shell command or `uses:` a reusable *action* pinned to a version. `if:` makes a job conditional; `permissions:` limits what the job's automatic `GITHUB_TOKEN` can do; `secrets.*` are encrypted values that never appear in logs.

Our pipeline: **lint → test (+ quality gate) → build and test the image → publish only for version tags**.

```
  push / PR ──▶ lint ──▶ test + quality gate ──▶ docker (tests in image, smoke test) ──▶ publish (only on tag v*)
```

In [ ]:
CI_WORKFLOW = """\
name: ci

on:
  push:
    branches: [main]
    tags: ["v*"]
  pull_request:

permissions:
  contents: read

concurrency:
  group: ci-${{ github.ref }}
  cancel-in-progress: true

jobs:
  lint:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v7
      - uses: astral-sh/setup-uv@v10.1.0
      - name: Ruff
        run: uvx ruff@0.16.7 check .

  test:
    runs-on: ubuntu-latest
    needs: lint
    steps:
      - uses: actions/checkout@v7
      - uses: actions/setup-python@v7
        with:
          python-version: "3.12"
      - uses: astral-sh/setup-uv@v10.1.0
      - name: Install dependencies
        run: uv pip install --system -r requirements-dev.txt
      - name: Data validation + model quality gate
        run: python scripts/quality_gate.py --model-dir model --data tests/fixtures/holdout.csv --config quality_gate.json
      - name: Unit, data, model and API tests
        run: python -m pytest -q

  docker:
    runs-on: ubuntu-latest
    needs: test
    steps:
      - uses: actions/checkout@v7
      - uses: docker/setup-buildx-action@v4
      - name: Run the test suite inside the image
        run: |
          docker build --target test -t housing-api:test .
          docker run --rm housing-api:test
      - name: Build the runtime image
        uses: docker/build-push-action@v7
        with:
          context: .
          target: runtime
          load: true
          tags: housing-api:ci
          cache-from: type=gha
          cache-to: type=gha,mode=max
      - name: Smoke test the container
        run: |
          docker run -d --name api -p 8000:8000 -e API_KEYS=ci-only-key housing-api:ci
          for i in $(seq 1 30); do curl -fsS localhost:8000/ready && break; sleep 1; done
          curl -fsS -X POST localhost:8000/v1/predict -H "X-API-Key: ci-only-key" -H "Content-Type: application/json" -d @tests/fixtures/smoke_row.json

  publish:
    runs-on: ubuntu-latest
    needs: docker
    if: startsWith(github.ref, 'refs/tags/v')
    permissions:
      contents: read
      packages: write
    steps:
      - uses: actions/checkout@v7
      - uses: docker/setup-buildx-action@v4
      - uses: docker/login-action@v4
        with:
          registry: ghcr.io
          username: ${{ github.actor }}
          password: ${{ secrets.GITHUB_TOKEN }}
      - id: meta
        uses: docker/metadata-action@v6
        with:
          images: ghcr.io/${{ github.repository }}/housing-api
          tags: |
            type=semver,pattern={{version}}
            type=sha
      - uses: docker/build-push-action@v7
        with:
          context: .
          target: runtime
          push: true
          tags: ${{ steps.meta.outputs.tags }}
          labels: ${{ steps.meta.outputs.labels }}
"""
workflow_path = SERVICE / ".github" / "workflows" / "ci.yml"
workflow_path.parent.mkdir(parents=True, exist_ok=True)
workflow_path.write_text(CI_WORKFLOW)

workflow = yaml.safe_load(workflow_path.read_text())
print("top-level keys as PyYAML parsed them:", list(workflow.keys()))

Look closely: the key `on` came back as the boolean **`True`**. YAML 1.1 (which PyYAML implements) treats `on`/`off`/`yes`/`no` as booleans. GitHub's own parser handles it, but any Python tooling you write around workflows must look up `workflow[True]`.

Now a structural validator for the kind of mistakes that break pipelines or leak credentials: missing jobs in `needs`, cycles, unpinned actions, publishing without a tag condition, over-broad permissions, and secrets pasted into YAML.

In [ ]:
PINNED_ACTION = re.compile(r"^[\w.-]+/[\w.-]+(/[\w./-]+)?@(v\d+(\.\d+)*|[0-9a-f]{40})$")
SECRET_LITERAL = re.compile(r"(sk-[A-Za-z0-9]{16,}|ghp_[A-Za-z0-9]{20,}|hf_[A-Za-z0-9]{20,}|AKIA[0-9A-Z]{16})")


def job_order(jobs):
    """Topological order of jobs by `needs`; raises ValueError on unknown jobs or cycles."""
    needs = {name: ([job.get("needs")] if isinstance(job.get("needs"), str) else job.get("needs", [])) for name, job in jobs.items()}
    order, done, visiting = [], set(), set()

    def visit(name, chain=()):
        if name not in jobs:
            raise ValueError(f"'{chain[-1]}' needs unknown job '{name}'")
        if name in done:
            return
        if name in visiting:
            raise ValueError(f"cycle: {' → '.join([*chain, name])}")
        visiting.add(name)
        for dependency in needs[name]:
            visit(dependency, (*chain, name))
        visiting.discard(name)
        done.add(name)
        order.append(name)

    for name in jobs:
        visit(name, (name,))
    return order


def validate_workflow(wf, raw_text=""):
    problems = []
    triggers = wf.get("on", wf.get(True))
    if not triggers:
        problems.append("no triggers (`on:`)")
    jobs = wf.get("jobs", {})
    try:
        order = job_order(jobs)
    except ValueError as err:
        problems.append(str(err))
        order = []
    top_permissions = wf.get("permissions", {})
    if top_permissions != {"contents": "read"}:
        problems.append(f"top-level permissions should be least-privilege {{contents: read}}, got {top_permissions}")
    for name, job in jobs.items():
        if "runs-on" not in job:
            problems.append(f"job '{name}' has no runs-on")
        for i, step in enumerate(job.get("steps", [])):
            if ("uses" in step) == ("run" in step):
                problems.append(f"job '{name}' step {i} must have exactly one of uses/run")
            if "uses" in step and not PINNED_ACTION.match(step["uses"]):
                problems.append(f"job '{name}' uses an unpinned action: {step['uses']}")
            pushes = step.get("with", {}).get("push") is True
            if pushes and "refs/tags/" not in str(job.get("if", "")):
                problems.append(f"job '{name}' pushes images without a tag-only `if:` condition")
        if job.get("permissions", {}).get("packages") == "write" and "refs/tags/" not in str(job.get("if", "")):
            problems.append(f"job '{name}' can write packages on every run")
    if SECRET_LITERAL.search(raw_text):
        problems.append("a literal secret appears in the workflow text")
    return order, problems


order, problems = validate_workflow(workflow, CI_WORKFLOW)
print("job execution order:", " → ".join(order))
print("problems:", problems or "none ✅")

broken = yaml.safe_load(CI_WORKFLOW)
broken["jobs"]["docker"]["needs"] = "tests"                                  # typo in a job name
broken["jobs"]["lint"]["steps"][0]["uses"] = "actions/checkout@main"         # moving branch, not a version
del broken["jobs"]["publish"]["if"]                                          # would publish on every push
broken_text = CI_WORKFLOW.replace("API_KEYS=ci-only-key", "API_KEYS=sk-live" + "A1b2C3d4E5f6G7h8I9")
_, broken_problems = validate_workflow(broken, broken_text)
print("\nbroken copy →")
for problem in broken_problems:
    print("  ❌", problem)

In [ ]:
# Does every pinned action version actually exist on GitHub? (a real network check)
actions_used = sorted({step["uses"] for job in workflow["jobs"].values() for step in job["steps"] if "uses" in step})
try:
    for ref in actions_used:
        repo, version = ref.split("@")
        status = httpx.get(f"https://github.com/{repo}/tree/{version}", timeout=20, follow_redirects=True).status_code
        print(f"{'✅' if status == 200 else '❌'} {ref} (HTTP {status})")
except httpx.HTTPError as err:
    print(f"⏭️ Skipped the GitHub check: no network ({type(err).__name__}).")

We can't run GitHub's hosted runners from a notebook, but we *can* run the same commands locally, which is what you should do before pushing. The lint step (tests and the quality gate come in the next two sections):

In [ ]:
if UV is None:
    print("⏭️ Skipped: needs uv to run `uvx ruff@0.16.7 check .` (install uv, or `pip install ruff==0.16.7` and run `ruff check .`).")
else:
    lint = run([UV, "tool", "run", "ruff@0.16.7", "check", "."], cwd=SERVICE, env={"NO_COLOR": "1", "UV_NO_PROGRESS": "1"}, max_lines=15)
    print("lint:", "passed ✅" if lint.returncode == 0 else f"failed (exit {lint.returncode})")

### ✍️ Your Turn

Write `get_triggers(workflow_dict)` that returns the **sorted list of event names** that trigger a workflow parsed by PyYAML, whether the key came back as `"on"` or as `True`.

In [ ]:
def get_triggers(wf):
    return None  # TODO


check("get_triggers", get_triggers(yaml.safe_load(CI_WORKFLOW)), ["pull_request", "push"],
      hint="triggers = wf.get('on', wf.get(True)); it can be a string, a list, or a dict.")

<details><summary>💡 Show solution</summary>

```python
def get_triggers(wf):
    triggers = wf.get("on", wf.get(True))
    if isinstance(triggers, str):
        return [triggers]
    return sorted(triggers)          # works for a list or a dict of events

check("get_triggers", get_triggers(yaml.safe_load(CI_WORKFLOW)), ["pull_request", "push"])
```
</details>

> 💡 **Interview angle:** "CI vs CD?" — CI merges and verifies every change automatically (lint, tests, build); continuous delivery always has a releasable artifact; continuous deployment ships it automatically. Mention pinned action versions (or SHAs), least-privilege `permissions`, OIDC instead of long-lived cloud keys, build caching, and publishing immutable, versioned images only from protected branches or tags.

## 12. Testing ML Systems: The Testing Pyramid 🟡

ML systems fail in more ways than normal software: the code can be right while the **data** is broken or the **model** is worse. The testing pyramid gets extra layers:

```
               ▲   end-to-end     container up → real HTTP → prediction matches offline model   (few, slow)
              ▲▲▲  API / integr.  TestClient with lifespan: auth, 422s, readiness, parity
            ▲▲▲▲▲  model          metric ≥ threshold on a fixed holdout; invariance & directional checks
          ▲▲▲▲▲▲▲  data           schema, nulls, ranges of the evaluation/training data
        ▲▲▲▲▲▲▲▲▲  unit           pure functions: feature logic, output formatting — no real model   (many, fast)
```

- **Invariance test:** a change that *shouldn't* matter doesn't change predictions (row order in a batch, an irrelevant field).
- **Directional test:** a change that *should* move predictions moves them the right way (higher income → higher average price).
- The holdout set is a **fixed, versioned file**. If it changed every run, metric changes would be noise.

In [ ]:
holdout_rows = X_test[:1000]
holdout = np.column_stack([holdout_rows, y_test[:1000]])
fixtures = SERVICE / "tests" / "fixtures"
fixtures.mkdir(parents=True, exist_ok=True)
np.savetxt(fixtures / "holdout.csv", holdout, delimiter=",", header=",".join([*FEATURES, "MedHouseVal"]), comments="", fmt="%.6g")
(fixtures / "smoke_row.json").write_text(json.dumps(sample_row))
print(f"holdout.csv: {len(holdout)} real test rows, {(fixtures / 'holdout.csv').stat().st_size / 1e3:.0f} kB")

write(SERVICE / "quality_gate.json", json.dumps({"min_rows": 500, "max_mae": 0.40, "min_improvement_over_baseline": 0.5,
                                                  "max_regression_vs_production": 0.02}, indent=2))

write(SERVICE / "tests" / "conftest.py", '''
    import csv
    import os
    from pathlib import Path

    import pytest

    ROOT = Path(__file__).resolve().parents[1]
    TEST_KEY = "test-key-not-a-secret"


    @pytest.fixture(scope="session")
    def holdout():
        with open(ROOT / "tests" / "fixtures" / "holdout.csv", newline="") as f:
            return [{k: float(v) for k, v in row.items()} for row in csv.DictReader(f)]


    @pytest.fixture(scope="session")
    def client():
        from fastapi.testclient import TestClient

        from app.main import app

        os.environ["MODEL_DIR"] = str(ROOT / "model")
        os.environ["API_KEYS"] = TEST_KEY
        with TestClient(app) as test_client:
            yield test_client
''')

write(SERVICE / "tests" / "test_unit.py", '''
    """Unit tests: output formatting and range flags, with a stub model (no file I/O, milliseconds)."""
    import numpy as np

    from app.model import ModelService


    class StubModel:
        def predict(self, X):
            return np.asarray(X)[:, 0] / 10          # "price" = income / 10, easy to reason about


    FEATURES = ["MedInc", "HouseAge"]
    META = {"model_version": "stub", "features": FEATURES, "training_ranges": {"MedInc": [1, 10], "HouseAge": [0, 50]}}


    def test_predict_formats_and_rounds():
        service = ModelService(StubModel(), META)
        assert service.predict([{"HouseAge": 20, "MedInc": 5.123456}]) == [{"median_house_value": 0.5123, "out_of_range_features": []}]


    def test_out_of_range_features_are_flagged():
        out = ModelService(StubModel(), META).predict([{"MedInc": 12, "HouseAge": -1}])
        assert out[0]["out_of_range_features"] == ["MedInc", "HouseAge"]


    def test_feature_order_comes_from_metadata_not_dict_order():
        a = ModelService(StubModel(), META).predict([{"MedInc": 3.0, "HouseAge": 40}])
        b = ModelService(StubModel(), META).predict([{"HouseAge": 40, "MedInc": 3.0}])
        assert a == b
''')

write(SERVICE / "tests" / "test_data.py", '''
    """Data tests: the evaluation data must match the serving schema."""
    import math

    from app.schemas import HouseFeatures

    EXPECTED_COLUMNS = ["MedInc", "HouseAge", "AveRooms", "AveBedrms", "Population", "AveOccup", "Latitude", "Longitude", "MedHouseVal"]


    def test_columns_and_size(holdout):
        assert list(holdout[0]) == EXPECTED_COLUMNS
        assert len(holdout) >= 500


    def test_no_missing_or_infinite_values(holdout):
        assert all(math.isfinite(value) for row in holdout for value in row.values())


    def test_every_row_passes_the_api_schema(holdout):
        for row in holdout:
            HouseFeatures.model_validate({k: v for k, v in row.items() if k != "MedHouseVal"})


    def test_target_range_matches_dataset_definition(holdout):
        assert all(0.14 <= row["MedHouseVal"] <= 5.00002 for row in holdout)
''')

write(SERVICE / "tests" / "test_model.py", '''
    """Model tests: quality threshold, invariance, and a directional check on the real model."""
    import json
    import random
    from pathlib import Path

    import numpy as np
    import pytest

    from app.model import ModelService

    ROOT = Path(__file__).resolve().parents[1]
    GATE = json.loads((ROOT / "quality_gate.json").read_text())


    @pytest.fixture(scope="module")
    def service():
        return ModelService.load(ROOT / "model")


    def features_only(rows):
        return [{k: v for k, v in row.items() if k != "MedHouseVal"} for row in rows]


    def test_mae_meets_quality_gate(service, holdout):
        predictions = np.array([p["median_house_value"] for p in service.predict(features_only(holdout))])
        truth = np.array([row["MedHouseVal"] for row in holdout])
        assert np.abs(predictions - truth).mean() <= GATE["max_mae"]


    def test_batch_order_does_not_change_predictions(service, holdout):
        rows = features_only(holdout[:200])
        shuffled = rows[:]
        random.Random(0).shuffle(shuffled)
        by_row = {json.dumps(r, sort_keys=True): p for r, p in zip(rows, service.predict(rows))}
        assert all(by_row[json.dumps(r, sort_keys=True)] == p for r, p in zip(shuffled, service.predict(shuffled)))


    def mean_prediction(predictions):
        return np.mean([p["median_house_value"] for p in predictions])


    def test_higher_income_raises_average_prediction(service, holdout):
        rows = features_only(holdout[:300])
        richer = [{**row, "MedInc": min(row["MedInc"] * 1.5, 25)} for row in rows]
        assert mean_prediction(service.predict(richer)) > mean_prediction(service.predict(rows))
''')

write(SERVICE / "tests" / "test_api.py", '''
    """API tests: the service contract, through the FastAPI app with its lifespan."""
    import json
    from pathlib import Path

    import pytest

    ROW = json.loads((Path(__file__).parent / "fixtures" / "smoke_row.json").read_text())
    AUTH = {"X-API-Key": "test-key-not-a-secret"}


    def test_health_and_ready(client):
        assert client.get("/health").status_code == 200
        assert client.get("/ready").json()["status"] == "ready"


    def test_requires_api_key(client):
        assert client.post("/v1/predict", json=ROW).status_code == 401


    @pytest.mark.parametrize("bad", [{}, {**ROW, "Latitude": 99.0}, {**ROW, "MedInc": "high"}, {**ROW, "extra": 1}])
    def test_invalid_input_is_422(client, bad):
        assert client.post("/v1/predict", json=bad, headers=AUTH).status_code == 422


    def test_batch_limits(client):
        assert client.post("/v1/predict/batch", json={"rows": []}, headers=AUTH).status_code == 422
        assert client.post("/v1/predict/batch", json={"rows": [ROW] * 257}, headers=AUTH).status_code == 422


    def test_single_and_batch_agree(client):
        single = client.post("/v1/predict", json=ROW, headers=AUTH).json()["median_house_value"]
        batch = client.post("/v1/predict/batch", json={"rows": [ROW, ROW]}, headers=AUTH).json()["predictions"]
        assert [p["median_house_value"] for p in batch] == [single, single]
''')

write(SERVICE / "pyproject.toml", '''
    [tool.pytest.ini_options]
    pythonpath = ["."]
    testpaths = ["tests"]
    addopts = "-p no:cacheprovider"
    filterwarnings = [
        "error",
        "ignore:The anyio.abc.BlockingPortal alias is deprecated:DeprecationWarning",
    ]

    [tool.ruff]
    line-length = 140
    target-version = "py312"
    extend-exclude = ["notebooks"]

    [tool.ruff.lint]
    # explicit rule set, so every machine and CI runner lints the same way
    select = ["E4", "E7", "E9", "F", "I", "RUF100"]
''')

In [ ]:
pyramid = []
for layer, test_file in [("unit", "test_unit.py"), ("data", "test_data.py"), ("model", "test_model.py"), ("API", "test_api.py")]:
    result = run([sys.executable, "-m", "pytest", f"tests/{test_file}", "-q", "--color=no", "--no-header"], cwd=SERVICE, show=False)
    summary = result.stdout.strip().splitlines()[-1]
    passed = int(re.search(r"(\d+) passed", summary).group(1)) if "passed" in summary else 0
    pyramid.append((layer, passed, result.seconds, result.returncode))
    print(f"{layer:6} {test_file:16} → {summary}")

assert all(code == 0 for *_, code in pyramid), "every test layer must pass"
fastest, slowest = min(pyramid, key=lambda p: p[2]), max(pyramid, key=lambda p: p[2])
print(f"\n{sum(p[1] for p in pyramid)} tests passed; {fastest[0]} tests ran in {fastest[2]:.1f} s vs {slowest[0]} tests in {slowest[2]:.1f} s "
      "(including interpreter start-up) — cheap tests at the bottom, expensive ones at the top.")

> 💡 **Interview angle:** "How do you test an ML system?" — the pyramid: unit tests for feature and formatting code with stub models; data tests on schemas, nulls, and ranges; model tests with metric thresholds on a fixed holdout plus invariance and directional (behavioural) checks; API contract tests; a few end-to-end container tests. In production, add monitoring for drift and live metrics, since tests only cover what you anticipated.

## 13. ML Quality Gates in CI 🔴

A **quality gate** is a CI step that **fails the pipeline** (non-zero exit code) when the release candidate isn't good enough, just like a failing unit test. For ML it checks two things:

1. **Data validation first.** If the evaluation data is broken (missing columns, NaNs, out-of-range values), any metric computed from it is meaningless.
2. **Model quality.** An absolute threshold (MAE ≤ 0.40), improvement over a trivial baseline, and **no regression versus the model currently in production** (≤ 2% worse).

The gate writes a JSON report (keep it as a CI artifact) and exits 1 on failure.

In [ ]:
write(SERVICE / "scripts" / "quality_gate.py", '''
    """CI quality gate: validate evaluation data, then check model metrics. Exit code 1 blocks the release."""
    import argparse
    import csv
    import json
    import math
    import sys
    from pathlib import Path

    import numpy as np
    from pydantic import ValidationError

    sys.path.insert(0, str(Path(__file__).resolve().parents[1]))     # make the `app` package importable from scripts/
    from app.model import ModelService
    from app.schemas import HouseFeatures

    TARGET = "MedHouseVal"


    def validate_data(path, features, min_rows):
        problems, rows = [], []
        with open(path, newline="") as f:
            reader = csv.DictReader(f)
            missing = [c for c in [*features, TARGET] if c not in (reader.fieldnames or [])]
            if missing:
                return [f"missing columns: {missing}"], []
            for i, raw in enumerate(reader, start=2):
                try:
                    row = {k: float(v) for k, v in raw.items()}
                except ValueError:
                    problems.append(f"line {i}: non-numeric value")
                    continue
                if not all(math.isfinite(v) for v in row.values()):
                    problems.append(f"line {i}: NaN or infinite value")
                    continue
                try:
                    HouseFeatures.model_validate({k: row[k] for k in features})
                except ValidationError as err:
                    problems.append(f"line {i}: {err.errors()[0]['loc'][0]} {err.errors()[0]['msg']}")
                    continue
                rows.append(row)
        if len(rows) < min_rows:
            problems.append(f"only {len(rows)} valid rows (need {min_rows})")
        return problems[:10] + ([f"... {len(problems) - 10} more"] if len(problems) > 10 else []), rows


    def main():
        parser = argparse.ArgumentParser()
        parser.add_argument("--model-dir", required=True)
        parser.add_argument("--data", required=True)
        parser.add_argument("--config", required=True)
        parser.add_argument("--production-metrics", help="metadata.json of the model currently in production")
        parser.add_argument("--report", default="gate_report.json")
        args = parser.parse_args()

        config = json.loads(Path(args.config).read_text())
        service = ModelService.load(args.model_dir)
        report = {"model_version": service.version, "checks": []}

        def record(name, passed, detail):
            report["checks"].append({"check": name, "passed": bool(passed), "detail": detail})

        data_problems, rows = validate_data(args.data, service.metadata["features"], config["min_rows"])
        record("data_validation", not data_problems, data_problems or f"{len(rows)} rows valid")

        if not data_problems:
            truth = np.array([r[TARGET] for r in rows])
            predictions = np.array([p["median_house_value"] for p in service.predict(rows)])
            mae = float(np.abs(predictions - truth).mean())
            baseline = float(np.abs(truth - truth.mean()).mean())
            improvement = 1 - mae / baseline
            report["metrics"] = {"mae": round(mae, 4), "baseline_mae": round(baseline, 4), "improvement": round(improvement, 4)}
            record("max_mae", mae <= config["max_mae"], f"{mae:.4f} <= {config['max_mae']}")
            record("beats_baseline", improvement >= config["min_improvement_over_baseline"],
                   f"improvement {improvement:.1%} >= {config['min_improvement_over_baseline']:.0%}")
            if args.production_metrics:
                production_mae = json.loads(Path(args.production_metrics).read_text())["metrics"]["holdout_mae"]
                allowed = production_mae * (1 + config["max_regression_vs_production"])
                record("no_regression_vs_production", mae <= allowed, f"{mae:.4f} <= {allowed:.4f} (production {production_mae:.4f})")

        report["passed"] = all(c["passed"] for c in report["checks"])
        Path(args.report).write_text(json.dumps(report, indent=2))
        for c in report["checks"]:
            print(f"{'PASS' if c['passed'] else 'FAIL'}  {c['check']:30} {c['detail']}")
        print("QUALITY GATE:", "PASSED" if report["passed"] else "FAILED")
        return 0 if report["passed"] else 1


    if __name__ == "__main__":
        sys.exit(main())
''')

# The model in "production" is v1: record its holdout MAE so candidates can be compared against it
production_metrics = WORK / "production_metadata.json"
v1_holdout_mae = float(np.abs(model_v1.predict(holdout_rows) - y_test[:1000]).mean())
production_metrics.write_text(json.dumps({"model_version": "housing-hgb-1.0.0", "metrics": {"holdout_mae": round(v1_holdout_mae, 4)}}))
print(f"production model v1 holdout MAE: {v1_holdout_mae:.4f}")


def gate(model_dir, data=fixtures / "holdout.csv", label=""):
    report_path = WORK / f"gate_report_{label}.json"
    result = run([sys.executable, "scripts/quality_gate.py", "--model-dir", model_dir, "--data", data, "--config", "quality_gate.json",
                  "--production-metrics", production_metrics, "--report", report_path], cwd=SERVICE, show=False)
    print(f"── {label} ── exit code {result.returncode}")
    print(textwrap.indent(result.stdout.strip() or result.stderr.strip()[-800:], "   "))
    return result.returncode


candidates = WORK / "candidates"
# 1) the real model
gate_results = {"v1 (current model)": gate(SERVICE / "model", label="v1")}

# 2) a label-misalignment bug: rows and targets shuffled independently (a classic bad-join bug)
shuffled_y = np.random.default_rng(SEED).permutation(y_train)
save_model(HistGradientBoostingRegressor(max_iter=300, random_state=SEED).fit(X_train, shuffled_y), candidates / "misaligned", "housing-hgb-bug", X_train, {})
gate_results["label-misalignment bug"] = gate(candidates / "misaligned", label="misaligned")

# 3) an under-trained candidate: passes the absolute threshold, but is worse than production
save_model(HistGradientBoostingRegressor(max_iter=40, random_state=SEED).fit(X_train, y_train), candidates / "undertrained", "housing-hgb-1.0.1", X_train, {})
gate_results["under-trained candidate"] = gate(candidates / "undertrained", label="undertrained")

# 4) the real model, but the evaluation data got corrupted upstream
corrupted = (fixtures / "holdout.csv").read_text().splitlines()
corrupted[5] = ",".join(["nan"] + corrupted[5].split(",")[1:])
corrupted[9] = corrupted[9].replace(corrupted[9].split(",")[6], "71.5", 1)      # latitude far outside California
bad_data = WORK / "holdout_corrupted.csv"
bad_data.write_text("\n".join(corrupted) + "\n")
gate_results["corrupted evaluation data"] = gate(SERVICE / "model", data=bad_data, label="corrupted")

print("\nsummary:", {name: "PASS" if code == 0 else "BLOCKED" for name, code in gate_results.items()})
assert gate_results["v1 (current model)"] == 0 and all(code == 1 for name, code in gate_results.items() if name != "v1 (current model)")

Each failure mode was caught by a *different* check: the misaligned labels by the absolute threshold and the baseline, the under-trained model only by the **comparison with production**, and the corrupted file by **data validation** before any metric was trusted. The gate runs in the `test` job of the workflow from Section 11.

> 💡 **Interview angle:** "How do you stop a worse model from being deployed?" — an automated gate in CI/CD: validate the evaluation data, compare against absolute thresholds **and** the current production model on the same fixed holdout (plus slice metrics for important segments and fairness), store the report, and require it before promotion. Offline gates don't replace an online canary, because live traffic can differ from the holdout.

## 14. Deployment Strategies and Rollback 🔴

| Strategy | How it works | Pros | Cons |
|---|---|---|---|
| **Recreate** | stop v1, start v2 | simple | downtime |
| **Rolling** | replace instances a few at a time (Kubernetes default) | no downtime, no extra capacity | v1 and v2 serve at the same time; rollback is another slow roll |
| **Blue/green** | run v2 (green) beside v1 (blue), test it, switch **all** traffic at once | instant switch and instant rollback | 2× capacity during release; all users hit v2 at once |
| **Canary** | send a small % of real traffic to v2, compare metrics, then increase | limits blast radius; real-traffic evidence | needs good metrics, and enough traffic for significance |
| **Shadow** (mirroring) | v2 receives a copy of traffic, but its answers are **not** returned | zero user impact; compare predictions safely | 2× compute; can't measure user-facing outcomes (clicks, conversions) |
| **A/B test** | split users to measure a business outcome | answers "is v2 better for the business?" | slow; needs experiment design |

**Rolling back a bad model** should be one boring action: redeploy the **previous immutable image digest** or move the model-registry alias back (`production → v1`), with automated triggers (error rate, latency, prediction-distribution alarms) and no schema changes that make v1 incompatible.

Let's run a **real canary** with containers: v1 (the baked-in model) and a candidate container serving the mounted under-trained model from Section 13. A tiny router in Python plays the load balancer, sending ~25% of 2,000 requests to the candidate by hashing the request ID. For the demo the true house values are known immediately. In production labels often arrive later, so canaries also watch proxy metrics (latency, errors, prediction distribution).

In [ ]:
def route(request_id, canary_percent):
    """Deterministic traffic split: the same request id always goes to the same version."""
    return "canary" if int(hashlib.sha256(request_id.encode()).hexdigest(), 16) % 100 < canary_percent else "stable"


traffic = [(f"req-{i}", dict(zip(FEATURES, map(float, X_test[i]))), float(y_test[i])) for i in range(1000, 3000)]

if not DOCKER_OK:
    docker_skip("the live canary with two containers")
else:
    stable_name, canary_name = f"stable-{RUN_ID}", f"canary-{RUN_ID}"
    stable_port = docker_run_service("housing-api:optimized", stable_name, env={"API_KEYS": API_KEY})
    canary_port = docker_run_service("housing-api:optimized", canary_name, env={"API_KEYS": API_KEY, "MODEL_DIR": "/models/candidate"},
                                     volumes=[f"{candidates / 'undertrained'}:/models/candidate:ro"])
    ports = {"stable": stable_port, "canary": canary_port}
    for arm, port in ports.items():
        wait_until_ready(f"http://127.0.0.1:{port}/ready", container=stable_name if arm == "stable" else canary_name)

    errors = {"stable": [], "canary": []}
    with httpx.Client(headers={"X-API-Key": API_KEY}) as http:
        versions = {arm: http.get(f"http://127.0.0.1:{port}/ready").json()["model_version"] for arm, port in ports.items()}
        for request_id, row, truth in traffic:
            arm = route(request_id, canary_percent=25)
            response = http.post(f"http://127.0.0.1:{ports[arm]}/v1/predict", json=row)
            errors[arm].append(abs(response.json()["median_house_value"] - truth))

    stable_mae, canary_mae = np.mean(errors["stable"]), np.mean(errors["canary"])
    rng = np.random.default_rng(SEED)
    boot = [rng.choice(errors["canary"], len(errors["canary"])).mean() - rng.choice(errors["stable"], len(errors["stable"])).mean() for _ in range(2000)]
    low, high = np.percentile(boot, [2.5, 97.5])
    decision = "ROLLBACK" if low > 0 and canary_mae > stable_mae * 1.02 else "PROMOTE" if high < 0.01 else "KEEP WATCHING"
    print(f"stable {versions['stable']}: {len(errors['stable'])} requests, MAE {stable_mae:.4f}")
    print(f"canary {versions['canary']}: {len(errors['canary'])} requests, MAE {canary_mae:.4f}")
    print(f"canary − stable MAE: {canary_mae - stable_mae:+.4f} (95% bootstrap CI {low:+.4f} … {high:+.4f}) → decision: {decision}")

    if decision == "ROLLBACK":
        remove_container(canary_name)            # rollback = stop routing to the candidate and remove it
        print(f"rolled back: canary removed; 100% of traffic stays on {versions['stable']}")
    remove_container(stable_name)
    remove_container(canary_name)

**Shadow deployment** answers a different question: *what would the candidate have said?* Every request is scored by both versions, users only ever see the stable answer, and we compare offline. It's safe even for a model you don't trust yet, and cheap to demonstrate in-process with the same real models:

In [ ]:
shadow_candidate = joblib.load(candidates / "undertrained" / "model.joblib")
shadow_X = X_test[1000:1600]
stable_pred, shadow_pred = model_v1.predict(shadow_X), shadow_candidate.predict(shadow_X)
delta = np.abs(shadow_pred - stable_pred)
print(f"shadow over {len(shadow_X)} requests: served MAE {mean_absolute_error(y_test[1000:1600], stable_pred):.4f} | "
      f"shadow MAE {mean_absolute_error(y_test[1000:1600], shadow_pred):.4f}")
print(f"|shadow − served| prediction difference: median {np.median(delta):.3f}, p95 {np.percentile(delta, 95):.3f} ($100k units); "
      f"{(delta > 0.25).mean():.0%} of requests differ by more than $25,000 — users saw none of it")

> 💡 **Interview angle:** "Blue/green vs canary?" — blue/green flips 100% of traffic between two full environments (fast switch and rollback, 2× capacity); canary ramps a small share of real traffic while comparing metrics (smaller blast radius, needs good metrics and time). "A new model is hurting conversions — what now?" — roll back first (previous image digest or registry alias), confirm metrics recover, then investigate offline (training/serving skew, data drift, a slice regression) and add the missing check to the quality gate.

## 🔧 Build It From Scratch

**Goal:** predict which Dockerfile steps will be rebuilt after a change, *without asking Docker*, then verify the prediction against real `docker build` output.

The cache-key recipe from Section 2, as code:

```
key(step 0)   = hash( FROM line )
key(step i)   = hash( key(step i-1)  +  instruction text  +  [for COPY/ADD] hash of every copied file's path, permissions, contents )
```

Files matching `.dockerignore` are never sent, so they can't affect any key, and modification times are *not* part of the hash. Because each key includes the previous key, one change ripples down to every later step. That's the "first miss rebuilds everything after it" rule, for free.

In [ ]:
SCRATCH = WORK / "cache_predictor"
shutil.rmtree(SCRATCH, ignore_errors=True)
write(SCRATCH / "requirements.txt", f"six==1.17.0  # run {RUN_ID}\n")
write(SCRATCH / "app" / "__init__.py", "")
write(SCRATCH / "app" / "main.py", f'RUN_ID = "{RUN_ID}"\nfrom app.utils import greet\n\nprint(greet())\n')
write(SCRATCH / "app" / "utils.py", 'def greet():\n    return "hello from the cache predictor demo"\n')
write(SCRATCH / "README.md", "# Cache predictor demo\n")
write(SCRATCH / ".dockerignore", "README.md\n*.log\n")
write(SCRATCH / "Dockerfile", f"""
    FROM {BASE_IMAGE}
    WORKDIR /srv
    COPY requirements.txt .
    RUN pip install --no-cache-dir -r requirements.txt
    COPY app/ ./app/
    RUN python -m compileall -q app
    CMD ["python", "-m", "app.main"]
""")


def ignore_patterns(context):
    path = Path(context) / ".dockerignore"
    lines = path.read_text().splitlines() if path.exists() else []
    return [line.strip() for line in lines if line.strip() and not line.strip().startswith("#")]


def pattern_to_regex(pattern):
    """.dockerignore rules (simplified): patterns are relative to the context root; `*` and `?` stay within one
    path segment; `**` matches any number of directories; a match on a directory excludes everything inside it."""
    pattern = pattern.strip("/")
    regex = ""
    i = 0
    while i < len(pattern):
        if pattern.startswith("**", i):
            regex += ".*"
            i += 2
            if pattern.startswith("/", i):
                regex += "/?"
                i += 1
        elif pattern[i] == "*":
            regex += "[^/]*"
            i += 1
        elif pattern[i] == "?":
            regex += "[^/]"
            i += 1
        else:
            regex += re.escape(pattern[i])
            i += 1
    return re.compile(f"{regex}(/.*)?")


def is_ignored(relative_path, patterns):
    return any(pattern_to_regex(p).fullmatch(relative_path) for p in patterns)


assert is_ignored("README.md", ["README.md"]) and not is_ignored("docs/README.md", ["README.md"])
assert is_ignored("model/model.joblib", ["**/*.joblib"]) and not is_ignored("model/model.joblib", ["*.joblib"])
assert is_ignored("data/raw/x.csv", ["data/"])


def copied_files(context, source, patterns):
    """Every file a COPY source expands to, after .dockerignore, in a stable order."""
    root = Path(context) / source
    files = [root] if root.is_file() else sorted(p for p in root.rglob("*") if p.is_file())
    return [f for f in files if not is_ignored(f.relative_to(context).as_posix(), patterns)]


def files_digest(context, files):
    h = hashlib.sha256()
    for f in files:
        h.update(f.relative_to(context).as_posix().encode())        # the path inside the context
        h.update(oct(stat.S_IMODE(f.stat().st_mode)).encode())      # permissions DO matter
        h.update(f.read_bytes())                                    # contents DO matter (mtime does NOT)
    return h.hexdigest()


def layer_keys(context):
    """[(instruction, cache key)] for every instruction in the Dockerfile."""
    context, patterns, keys, previous = Path(context), ignore_patterns(context), [], ""
    for line in (context / "Dockerfile").read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        inputs = ""
        if line.split()[0].upper() in ("COPY", "ADD") and "--from" not in line:
            sources = line.split()[1:-1]
            inputs = files_digest(context, [f for s in sources for f in copied_files(context, s, patterns)])
        previous = hashlib.sha256(f"{previous}|{line}|{inputs}".encode()).hexdigest()
        keys.append((line, previous))
    return keys


def predict_rebuilds(before, after):
    """Steps BuildKit executes (WORKDIR/COPY/ADD/RUN) whose key changed."""
    return [line for (line, old), (_, new) in zip(before, after)
            if old != new and line.split()[0].upper() in ("WORKDIR", "COPY", "ADD", "RUN")]


def docker_rebuilt(build):
    return [s["instruction"] for s in build.steps if not s["cached"] and not s["instruction"].startswith("FROM")]


scenarios = [
    ("touch app/main.py (modification time only)", lambda: os.utime(SCRATCH / "app" / "main.py")),
    ("edit README.md (listed in .dockerignore)", lambda: (SCRATCH / "README.md").write_text("# Cache predictor demo\n\nMore docs.\n")),
    ("edit app/utils.py", lambda: (SCRATCH / "app" / "utils.py").write_text('def greet():\n    return "hello, edited"\n')),
    ("chmod +x app/main.py (permissions only)", lambda: (SCRATCH / "app" / "main.py").chmod(0o755)),
    ("edit requirements.txt", lambda: (SCRATCH / "requirements.txt").write_text(f"six==1.17.0  # run {RUN_ID}, edited\n")),
]

if DOCKER_OK:
    first = docker_build(SCRATCH, "cache-predictor:demo", show_steps=False)
    print(f"initial build: {len(docker_rebuilt(first))} steps executed\n")
else:
    docker_skip("verifying the predictions against docker build")

results = []
for label, change in scenarios:
    before = layer_keys(SCRATCH)
    change()
    predicted = predict_rebuilds(before, layer_keys(SCRATCH))
    actual = docker_rebuilt(docker_build(SCRATCH, "cache-predictor:demo", show_steps=False)) if DOCKER_OK else None
    results.append((label, predicted, actual))
    verdict = "(not verified)" if actual is None else "✅ match" if predicted == actual else "❌ MISMATCH"
    print(f"{label}\n   predicted rebuild: {predicted or 'nothing'}\n   docker rebuilt   : {actual if actual is not None else '—'}  {verdict}")

if DOCKER_OK:
    assert all(p == a for _, p, a in results), "the predictor disagreed with Docker"
    print(f"\n✅ the from-scratch predictor matched real BuildKit behaviour in all {len(results)} scenarios")
    print("container output:", run([DOCKER, "run", "--rm", "cache-predictor:demo"], show=False).stdout.strip())

Lessons confirmed by real builds: touching a file, or editing an ignored file, costs **nothing**. Changing code rebuilds only from its `COPY` down. Changing permissions counts as a change. Changing the dependency list re-runs the expensive install. Real BuildKit keys also include the base image **digest**, build args, and more metadata, but this is the core idea.

## ⚠️ Common Pitfalls

### ❌ Pitfall 1 — Shell-form `CMD`: your app never hears "stop"

With the shell form (`CMD uvicorn …`), Docker runs `/bin/sh -c "…"` as **PID 1**. `docker stop` sends SIGTERM to PID 1, the shell doesn't forward it, and after the stop timeout Docker sends **SIGKILL**. In-flight requests are cut off and shutdown code never runs.

In [ ]:
PITFALLS = WORK / "pitfalls"
write(PITFALLS / "Dockerfile.shellform", """
    FROM housing-api:optimized
    # ❌ shell form: /bin/sh is PID 1
    CMD python -c "print('warming up')" && uvicorn app.main:app --host 0.0.0.0 --port 8000
""")


def stop_behaviour(image, label):
    name = f"stop-{image.split(':')[1]}-{RUN_ID}"          # container names allow only letters, digits, _ . -
    port = docker_run_service(image, name, env={"API_KEYS": API_KEY})
    wait_until_ready(f"http://127.0.0.1:{port}/ready", container=name)
    pid1 = run([DOCKER, "exec", name, "cat", "/proc/1/cmdline"], show=False).stdout.replace("\0", " ").strip()
    stop = run([DOCKER, "stop", "-t", "10", name], show=False)          # up to 10 s for a graceful exit, then SIGKILL
    code = run([DOCKER, "inspect", "-f", "{{.State.ExitCode}}", name], show=False).stdout.strip()
    graceful = "model unloaded" in run([DOCKER, "logs", name], show=False).stdout
    remove_container(name)
    print(f"{label:6} PID 1 = {pid1[:70]:70} | stop {stop.seconds:4.1f} s | exit {code:>3} | graceful shutdown ran: {graceful}")


if not DOCKER_OK:
    docker_skip("the stop-signal pitfall")
else:
    docker_build(PITFALLS, "pitfall:shellform", dockerfile=PITFALLS / "Dockerfile.shellform", show_steps=False)
    stop_behaviour("pitfall:shellform", "❌ shell")
    stop_behaviour("housing-api:optimized", "✅ exec")
    print("(`docker stop -t 10`: the shell-form container ignored SIGTERM, so Docker waited the full timeout and then sent SIGKILL)")

Fix: the JSON **exec form** `CMD ["uvicorn", …]`. If you really need a startup script, end it with `exec uvicorn …` so the server *replaces* the shell as PID 1.

### ❌ Pitfall 2 — Binding to `127.0.0.1` inside the container

In [ ]:
write(PITFALLS / "Dockerfile.localhost", """
    FROM housing-api:optimized
    # ❌ listens only on the container's own loopback interface
    CMD ["uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"]
""")

if not DOCKER_OK:
    docker_skip("the localhost-binding pitfall")
else:
    docker_build(PITFALLS, "pitfall:localhost", dockerfile=PITFALLS / "Dockerfile.localhost", show_steps=False)
    for label, image in [("❌ --host 127.0.0.1", "pitfall:localhost"), ("✅ --host 0.0.0.0", "housing-api:optimized")]:
        name = f"bind-{RUN_ID}-{image.split(':')[1]}"
        port = docker_run_service(image, name, env={"API_KEYS": API_KEY})
        outcome = "no answer"
        for _ in range(40):
            try:
                outcome = f"HTTP {httpx.get(f'http://127.0.0.1:{port}/ready', timeout=1).status_code}"
                break
            except httpx.TransportError as err:
                outcome = f"{type(err).__name__}"
                time.sleep(0.25)
        health = "starting"
        for _ in range(40):
            health = run([DOCKER, "inspect", "-f", "{{.State.Health.Status}}", name], show=False).stdout.strip()
            if health != "starting":
                break
            time.sleep(0.25)
        print(f"{label:20} from the host: {outcome:16} | Docker HEALTHCHECK (runs inside): {health}")
        remove_container(name)

The broken container even reports **healthy**, because its health check runs *inside* the container where `127.0.0.1` works. Only a check from outside (a smoke test) catches it.

### ❌ Pitfall 3 — Deleting files in a later layer doesn't shrink the image

In [ ]:
write(PITFALLS / "Dockerfile.twolayers", f"""
    FROM {BASE_IMAGE}
    RUN head -c 60000000 /dev/urandom > /tmp/weights.bin
    # ❌ the file still exists in the previous layer
    RUN rm /tmp/weights.bin
""")
write(PITFALLS / "Dockerfile.onelayer", f"""
    FROM {BASE_IMAGE}
    # ✅ create, use, and delete within ONE layer
    RUN head -c 60000000 /dev/urandom > /tmp/weights.bin && rm /tmp/weights.bin
""")

if not DOCKER_OK:
    docker_skip("the deleted-file pitfall")
else:
    base_disk, base_pull = image_size(BASE_IMAGE)
    for label, variant in [("❌ rm in a later RUN", "twolayers"), ("✅ rm in the same RUN", "onelayer")]:
        docker_build(PITFALLS, f"pitfall:{variant}", dockerfile=PITFALLS / f"Dockerfile.{variant}", show_steps=False)
        disk, pull = image_size(f"pitfall:{variant}")
        print(f"{label:22} adds {max(disk - base_disk, 0) / 1e6:5.0f} MB on disk and {max(pull - base_pull, 0) / 1e6:5.0f} MB to every pull")

The same applies to `apt-get install … && rm -rf /var/lib/apt/lists/*` (one `RUN`), downloaded model archives, and secrets. A later `rm` hides a file but never removes it from the image. Multi-stage builds avoid the problem entirely.

### ❌ Pitfall 4 — Tests pass on your laptop, but the image is broken

A teammate adds `**/*.joblib` to `.dockerignore` ("model files shouldn't be in the build"). Every test on the host still passes, because the host has the file. The image doesn't. (Note the `**/`: `.dockerignore` patterns are anchored at the context root, so a plain `*.joblib` would only match files at the top level.)

In [ ]:
shutil.copy(SERVICE / "Dockerfile", SERVICE / "Dockerfile.badignore")
(SERVICE / "Dockerfile.badignore.dockerignore").write_text((SERVICE / ".dockerignore").read_text() + "**/*.joblib\n")   # BuildKit reads <Dockerfile>.dockerignore

host_tests = run([sys.executable, "-m", "pytest", "tests/test_model.py", "-q", "--color=no", "--no-header"], cwd=SERVICE, show=False)
print("host-side model tests (look fine):", host_tests.stdout.strip().splitlines()[-1])

if not DOCKER_OK:
    docker_skip("the broken-image pitfall")
else:
    docker_build(SERVICE, "pitfall:badignore", dockerfile=SERVICE / "Dockerfile.badignore", target="runtime", show_steps=False)
    name = f"badignore-{RUN_ID}"
    docker_run_service("pitfall:badignore", name, env={"API_KEYS": API_KEY})
    time.sleep(4)
    status = run([DOCKER, "ps", "-a", "--filter", f"name={name}", "--format", "{{.Status}}"], show=False).stdout.strip()
    logs = run([DOCKER, "logs", name], show=False).stdout.strip().splitlines()
    print(f"❌ the container built from the same code: {status}")
    print("   log lines that explain it:", *[f"\n     {line[:140]}" for line in logs if "Error" in line or "startup failed" in line][-2:])
    remove_container(name)
    print("✅ fix: test the IMAGE, not just the code — the mini project runs the test suite inside the built image, plus a container smoke test in CI")
(SERVICE / "Dockerfile.badignore").unlink()
(SERVICE / "Dockerfile.badignore.dockerignore").unlink()

## 🏋️ Practice Exercises

Try each one before opening the solution. Run the cell: ⏳ means not attempted, ✅ means correct. None of the checkers need Docker.

### 🟢 Exercise 1 — Mutable tag, `latest`, or digest?
Write `classify_image_ref(ref)` returning `"digest"` if the reference is pinned by `@sha256:…`, `"latest"` if it has no tag or the tag is `latest`, and `"tag"` otherwise. Careful: `localhost:5000/api` has a colon that belongs to the *registry port*, not a tag.

In [ ]:
def classify_image_ref(ref):
    return None  # TODO


image_refs = ["python:3.12-slim", "python", "ghcr.io/acme/housing-api:latest",
              "python@sha256:78387bc3881b8273120a12ebe6c1ab22b018ccc2c9adf565ae1ac9b536e184ea",
              "localhost:5000/housing-api", "localhost:5000/housing-api:1.0.0"]
got = None if classify_image_ref(image_refs[0]) is None else [classify_image_ref(r) for r in image_refs]
check("classify_image_ref", got, ["tag", "latest", "latest", "digest", "latest", "tag"],
      hint="Check '@sha256:' first; then look for ':' only in the part after the last '/'.")

<details><summary>💡 Show solution</summary>

```python
def classify_image_ref(ref):
    if "@sha256:" in ref:
        return "digest"
    name = ref.rsplit("/", 1)[-1]            # the registry host (and its port) comes before the last '/'
    if ":" not in name or name.endswith(":latest"):
        return "latest"
    return "tag"

image_refs = ["python:3.12-slim", "python", "ghcr.io/acme/housing-api:latest",
              "python@sha256:78387bc3881b8273120a12ebe6c1ab22b018ccc2c9adf565ae1ac9b536e184ea",
              "localhost:5000/housing-api", "localhost:5000/housing-api:1.0.0"]
check("classify_image_ref", [classify_image_ref(r) for r in image_refs], ["tag", "latest", "latest", "digest", "latest", "tag"])
```

Tags are movable pointers: `python:3.12-slim` will point to a newer build next month. Digests are immutable, so production deploys (and rollbacks) should reference digests.
</details>

### 🟢 Exercise 2 — Pick the deployment strategy
Answer with one of: `"recreate"`, `"rolling"`, `"blue/green"`, `"canary"`, `"shadow"`.

In [ ]:
strategy_for = {
    "a nightly batch-scoring job where a few minutes of downtime are fine and simplicity wins": None,
    "the Kubernetes default: replace pods a few at a time without a second environment": None,
    "switch 100% of traffic at once, keep the old environment running for an instant switch back": None,
    "send 5% of real users to the new model, and increase only if metrics hold": None,
    "score live traffic with a risky new model without any user ever seeing its answers": None,
}
check("strategy_for", None if None in strategy_for.values() else strategy_for,
      {"a nightly batch-scoring job where a few minutes of downtime are fine and simplicity wins": "recreate",
       "the Kubernetes default: replace pods a few at a time without a second environment": "rolling",
       "switch 100% of traffic at once, keep the old environment running for an instant switch back": "blue/green",
       "send 5% of real users to the new model, and increase only if metrics hold": "canary",
       "score live traffic with a risky new model without any user ever seeing its answers": "shadow"},
      hint="Re-read the table at the top of Section 14.")

<details><summary>💡 Show solution</summary>

```python
strategy_for = {
    "a nightly batch-scoring job where a few minutes of downtime are fine and simplicity wins": "recreate",
    "the Kubernetes default: replace pods a few at a time without a second environment": "rolling",
    "switch 100% of traffic at once, keep the old environment running for an instant switch back": "blue/green",
    "send 5% of real users to the new model, and increase only if metrics hold": "canary",
    "score live traffic with a risky new model without any user ever seeing its answers": "shadow",
}
check("strategy_for", strategy_for,
      {"a nightly batch-scoring job where a few minutes of downtime are fine and simplicity wins": "recreate",
       "the Kubernetes default: replace pods a few at a time without a second environment": "rolling",
       "switch 100% of traffic at once, keep the old environment running for an instant switch back": "blue/green",
       "send 5% of real users to the new model, and increase only if metrics hold": "canary",
       "score live traffic with a risky new model without any user ever seeing its answers": "shadow"})
```
</details>

### 🟢 Exercise 3 — Exec form or shell form?
Write `cmd_form(line)` that returns `"exec"` when the arguments after `CMD`/`ENTRYPOINT` are a **valid JSON array of strings**, else `"shell"`. (Docker's rule: anything that isn't valid JSON, including single-quoted "arrays", is shell form.)

In [ ]:
def cmd_form(line):
    return None  # TODO


cmd_lines = ['CMD ["uvicorn", "app.main:app"]', "CMD uvicorn app.main:app", 'ENTRYPOINT ["python", "-m", "app"]',
             "CMD ['uvicorn', 'app.main:app']", 'CMD ["python", "serve.py"]   ']
got = None if cmd_form(cmd_lines[0]) is None else [cmd_form(line) for line in cmd_lines]
check("cmd_form", got, ["exec", "shell", "exec", "shell", "exec"],
      hint="Split off the instruction with line.split(maxsplit=1), then try json.loads on the rest.")

<details><summary>💡 Show solution</summary>

```python
def cmd_form(line):
    arguments = line.split(maxsplit=1)[1].strip()
    try:
        parsed = json.loads(arguments)
    except json.JSONDecodeError:
        return "shell"
    return "exec" if isinstance(parsed, list) and all(isinstance(x, str) for x in parsed) else "shell"

cmd_lines = ['CMD ["uvicorn", "app.main:app"]', "CMD uvicorn app.main:app", 'ENTRYPOINT ["python", "-m", "app"]',
             "CMD ['uvicorn', 'app.main:app']", 'CMD ["python", "serve.py"]   ']
check("cmd_form", [cmd_form(line) for line in cmd_lines], ["exec", "shell", "exec", "shell", "exec"])
```

The single-quoted version is the sneaky one: it *looks* like exec form, but Docker runs it through `/bin/sh -c`, bringing back Pitfall 1.
</details>

### 🟡 Exercise 4 — Which CI jobs run for this event?
Write `jobs_that_run(workflow, ref)` that returns the jobs that run, in execution order, for a Git ref such as `"refs/heads/main"` or `"refs/tags/v1.2.0"`. A job runs if its `if:` condition (only the form `startsWith(github.ref, '...')` needs support) is true or absent, **and** all the jobs it `needs` ran. You may use `job_order()` from Section 11.

In [ ]:
def jobs_that_run(wf, ref):
    return None  # TODO


ci = yaml.safe_load(CI_WORKFLOW)
got = None if jobs_that_run(ci, "refs/heads/main") is None else {"main": jobs_that_run(ci, "refs/heads/main"), "tag": jobs_that_run(ci, "refs/tags/v1.2.0")}
check("jobs_that_run", got, {"main": ["lint", "test", "docker"], "tag": ["lint", "test", "docker", "publish"]},
      hint=r"re.fullmatch(r\"startsWith\(github\.ref,\s*'([^']*)'\)\", condition) extracts the prefix.")

<details><summary>💡 Show solution</summary>

```python
def jobs_that_run(wf, ref):
    ran = []
    for name in job_order(wf["jobs"]):
        job = wf["jobs"][name]
        needs = job.get("needs", [])
        needs = [needs] if isinstance(needs, str) else needs
        condition_ok = True
        if "if" in job:
            match = re.fullmatch(r"startsWith\(github\.ref,\s*'([^']*)'\)", job["if"].strip())
            condition_ok = bool(match) and ref.startswith(match.group(1))
        if condition_ok and all(n in ran for n in needs):
            ran.append(name)
    return ran

ci = yaml.safe_load(CI_WORKFLOW)
check("jobs_that_run", {"main": jobs_that_run(ci, "refs/heads/main"), "tag": jobs_that_run(ci, "refs/tags/v1.2.0")},
      {"main": ["lint", "test", "docker"], "tag": ["lint", "test", "docker", "publish"]})
```

Real GitHub Actions also skips a job's dependents when that job is skipped, unless they use `if: always()`.
</details>

### 🟡 Exercise 5 — A quality-gate decision function
Write `evaluate_gate(candidate, production, rules)`, where `candidate = {"mae": float, "rows": int}`, `production = {"mae": float}`, and `rules = {"max_mae", "max_regression", "min_rows"}`. Return `{"passed": bool, "failed": sorted list of failed rule names}`:
- `max_mae`: fails if `candidate mae > max_mae`
- `max_regression`: fails if `candidate mae > production mae × (1 + max_regression)`
- `min_rows`: fails if `candidate rows < min_rows`

In [ ]:
def evaluate_gate(candidate, production, rules):
    return None  # TODO


gate_rules = {"max_mae": 0.40, "max_regression": 0.02, "min_rows": 500}
gate_cases = [({"mae": 0.30, "rows": 1000}, {"mae": 0.31}), ({"mae": 0.45, "rows": 1000}, {"mae": 0.31}), ({"mae": 0.32, "rows": 100}, {"mae": 0.30})]
got = None if evaluate_gate(*gate_cases[0], gate_rules) is None else [evaluate_gate(c, p, gate_rules) for c, p in gate_cases]
check("evaluate_gate", got,
      [{"passed": True, "failed": []}, {"passed": False, "failed": ["max_mae", "max_regression"]},
       {"passed": False, "failed": ["max_regression", "min_rows"]}],
      hint="Build a list of failed names, then return {'passed': not failed, 'failed': sorted(failed)}.")

<details><summary>💡 Show solution</summary>

```python
def evaluate_gate(candidate, production, rules):
    failed = []
    if candidate["mae"] > rules["max_mae"]:
        failed.append("max_mae")
    if candidate["mae"] > production["mae"] * (1 + rules["max_regression"]):
        failed.append("max_regression")
    if candidate["rows"] < rules["min_rows"]:
        failed.append("min_rows")
    return {"passed": not failed, "failed": sorted(failed)}

gate_rules = {"max_mae": 0.40, "max_regression": 0.02, "min_rows": 500}
gate_cases = [({"mae": 0.30, "rows": 1000}, {"mae": 0.31}), ({"mae": 0.45, "rows": 1000}, {"mae": 0.31}), ({"mae": 0.32, "rows": 100}, {"mae": 0.30})]
check("evaluate_gate", [evaluate_gate(c, p, gate_rules) for c, p in gate_cases],
      [{"passed": True, "failed": []}, {"passed": False, "failed": ["max_mae", "max_regression"]},
       {"passed": False, "failed": ["max_regression", "min_rows"]}])
```
</details>

### 🔴 Exercise 6 — Automated canary analysis (interview classic)
A canary served `canary_total` requests with `canary_errors` failures (5xx or timeouts); stable served `stable_total` with `stable_errors`. Write `canary_decision(stable_errors, stable_total, canary_errors, canary_total, alpha=0.05, min_requests=500)` returning `(decision, p_value)`:
- `("wait", None)` if the canary has fewer than `min_requests` requests
- otherwise run a **one-sided two-proportion z-test** (is the canary's error rate *higher*?): pooled rate `p = (e_s + e_c) / (n_s + n_c)`, `z = (p_c − p_s) / sqrt(p(1−p)(1/n_s + 1/n_c))`, `p_value = 1 − Φ(z)` with `Φ(z) = 0.5·(1 + erf(z/√2))`
- `("rollback", p_value)` if `p_value < alpha`, else `("promote", p_value)`. If there are no errors at all, return `("promote", 1.0)`
- round `p_value` to 4 decimals

In [ ]:
def canary_decision(stable_errors, stable_total, canary_errors, canary_total, alpha=0.05, min_requests=500):
    return None  # TODO


canary_cases = [(50, 10_000, 3, 400), (50, 10_000, 12, 1_000), (50, 10_000, 6, 1_000), (0, 5_000, 0, 800)]
got = None if canary_decision(*canary_cases[1]) is None else [canary_decision(*case) for case in canary_cases]
check("canary_decision", got, [("wait", None), ("rollback", 0.0024), ("promote", 0.3359), ("promote", 1.0)],
      hint="Compute rates, the pooled rate, z, then p = 1 - 0.5 * (1 + math.erf(z / math.sqrt(2))).")

<details><summary>💡 Show solution</summary>

```python
def canary_decision(stable_errors, stable_total, canary_errors, canary_total, alpha=0.05, min_requests=500):
    if canary_total < min_requests:
        return ("wait", None)
    pooled = (stable_errors + canary_errors) / (stable_total + canary_total)
    if pooled == 0:
        return ("promote", 1.0)
    p_stable, p_canary = stable_errors / stable_total, canary_errors / canary_total
    z = (p_canary - p_stable) / math.sqrt(pooled * (1 - pooled) * (1 / stable_total + 1 / canary_total))
    p_value = round(1 - 0.5 * (1 + math.erf(z / math.sqrt(2))), 4)
    return ("rollback" if p_value < alpha else "promote", p_value)

canary_cases = [(50, 10_000, 3, 400), (50, 10_000, 12, 1_000), (50, 10_000, 6, 1_000), (0, 5_000, 0, 800)]
check("canary_decision", [canary_decision(*case) for case in canary_cases],
      [("wait", None), ("rollback", 0.0024), ("promote", 0.3359), ("promote", 1.0)])
```

**Talking points:** 1.2% vs 0.5% errors is significant with 1,000 canary requests (p ≈ 0.002), while 0.6% vs 0.5% is not (p ≈ 0.34). Too little traffic means "wait", not "promote". Real canary analysis (Argo Rollouts, Flagger, Kayenta) also checks latency and business metrics, and repeated checks need corrections for peeking (sequential testing).
</details>

## 🚀 Mini Project: Release Pipeline for the Housing Model Service

**Goal:** take the housing price service from commit to a running, verified container, the way a CI pipeline would. One command per step, stopping at the first failure:

1. validate the workflow file → 2. lint → 3. data validation + model quality gate → 4. tests on the host → 5. build the **test** stage and run the test suite **inside the image** → 6. build the **runtime** image → 7. start it, wait for Docker's health check → 8. real predictions must match the offline model → 9. load test (p50/p95/p99) → 10. graceful stop.

Then report the measured image size, cold start, and latency, with conclusions computed from the numbers.

### Step 1 — The final multi-stage Dockerfile (builder → runtime → test)

In [ ]:
FINAL_DOCKERFILE = (SERVICE / "Dockerfile").read_text() + f"""
# ---------- stage 3: test — the exact runtime image plus test tools (never deployed) ----------
FROM runtime AS test
USER root
COPY --from={UV_IMAGE} /uv /bin/uv
COPY requirements.txt requirements-dev.txt ./
RUN uv pip install --python /opt/venv/bin/python --no-cache -r requirements-dev.txt
COPY pyproject.toml quality_gate.json ./
COPY scripts/ ./scripts/
COPY tests/ ./tests/
USER app
CMD ["python", "-m", "pytest", "-q", "--color=no"]
"""
(SERVICE / "Dockerfile").write_text(FINAL_DOCKERFILE)
print(FINAL_DOCKERFILE)
tree(SERVICE)

`docker build` without `--target` builds the **last** stage (here `test`), so the pipeline always names its target: `--target test` in CI, `--target runtime` for the image that ships.

### Step 2 — Run the pipeline

In [ ]:
import asyncio


async def latency_test(url, payloads, n_requests, concurrency, headers):
    latencies, statuses = [], []
    semaphore = asyncio.Semaphore(concurrency)
    async with httpx.AsyncClient(headers=headers, timeout=30, limits=httpx.Limits(max_connections=concurrency)) as client:
        await asyncio.gather(*(client.post(url, json=payloads[i]) for i in range(concurrency)))        # warm-up

        async def one(i):
            async with semaphore:
                start = time.perf_counter()
                response = await client.post(url, json=payloads[i % len(payloads)])
                latencies.append((time.perf_counter() - start) * 1000)
                statuses.append(response.status_code)

        start = time.perf_counter()
        await asyncio.gather(*(one(i) for i in range(n_requests)))
        elapsed = time.perf_counter() - start
    lat = np.array(latencies)
    return {"concurrency": concurrency, "errors": sum(s != 200 for s in statuses), "p50": np.percentile(lat, 50),
            "p95": np.percentile(lat, 95), "p99": np.percentile(lat, 99), "rps": n_requests / elapsed}


pipeline, facts = [], {}
RELEASE_TAG = "housing-api:1.0.0"
release_container = f"release-{RUN_ID}"


def record(name, status, seconds, detail):
    pipeline.append({"step": name, "status": status, "seconds": seconds, "detail": detail})
    icon = {"passed": "✅", "failed": "❌", "skipped": "⏭️", "not run": "·"}[status]
    print(f"{icon} {name:38} {seconds:6.1f} s  {detail}")


def blocked():
    return any(p["status"] == "failed" for p in pipeline)


async def run_pipeline():
    steps = []

    def step(name, needs_docker=False, needs_uv=False):
        def register(fn):
            steps.append((name, needs_docker, needs_uv, fn))
            return fn
        return register

    @step("1. validate CI workflow")
    async def s1():
        _, problems = validate_workflow(yaml.safe_load(workflow_path.read_text()), workflow_path.read_text())
        return not problems, problems or "structure, pins, permissions OK"

    @step("2. lint (ruff)", needs_uv=True)
    async def s2():
        result = run([UV, "tool", "run", "ruff@0.16.7", "check", "."], cwd=SERVICE, env={"NO_COLOR": "1"}, show=False)
        return result.returncode == 0, result.stdout.strip().splitlines()[-1]

    @step("3. data validation + quality gate")
    async def s3():
        result = run([sys.executable, "scripts/quality_gate.py", "--model-dir", "model", "--data", "tests/fixtures/holdout.csv",
                      "--config", "quality_gate.json", "--production-metrics", production_metrics, "--report", WORK / "release_gate.json"],
                     cwd=SERVICE, show=False)
        return result.returncode == 0, result.stdout.strip().splitlines()[-1]

    @step("4. tests on the host")
    async def s4():
        result = run([sys.executable, "-m", "pytest", "-q", "--color=no", "--no-header"], cwd=SERVICE, show=False)
        facts["host_tests"] = result.stdout.strip().splitlines()[-1]
        return result.returncode == 0, facts["host_tests"]

    @step("5. build test stage + tests in image", needs_docker=True)
    async def s5():
        docker_build(SERVICE, "housing-api:test", target="test", show_steps=False)
        result = run([DOCKER, "run", "--rm", "housing-api:test"], show=False)
        facts["image_tests"] = result.stdout.strip().splitlines()[-1]
        return result.returncode == 0, facts["image_tests"]

    @step("6. build runtime image", needs_docker=True)
    async def s6():
        build = docker_build(SERVICE, RELEASE_TAG, target="runtime", show_steps=False)
        facts["size"] = image_size(RELEASE_TAG)
        cached = sum(s["cached"] for s in build.steps)
        return True, f"{facts['size'][0] / 1e6:.0f} MB on disk, {facts['size'][1] / 1e6:.0f} MB to pull ({cached}/{len(build.steps)} steps cached)"

    @step("7. start container + health check", needs_docker=True)
    async def s7():
        start = time.perf_counter()
        facts["port"] = docker_run_service(RELEASE_TAG, release_container, env={"API_KEYS": API_KEY})
        wait_until_ready(f"http://127.0.0.1:{facts['port']}/ready", container=release_container)
        facts["cold_start"] = time.perf_counter() - start
        health = "starting"
        while health == "starting" and time.perf_counter() - start < 60:
            health = run([DOCKER, "inspect", "-f", "{{.State.Health.Status}}", release_container], show=False).stdout.strip()
            await asyncio.sleep(0.25)
        facts["healthy_after"] = time.perf_counter() - start
        return health == "healthy", f"ready in {facts['cold_start']:.2f} s, Docker status '{health}' after {facts['healthy_after']:.1f} s"

    @step("8. predictions match offline model", needs_docker=True)
    async def s8():
        rows = [dict(zip(FEATURES, map(float, r))) for r in X_test[:200]]
        body = httpx.post(f"http://127.0.0.1:{facts['port']}/v1/predict/batch", json={"rows": rows}, headers={"X-API-Key": API_KEY}).json()
        diff = np.abs(np.array([p["median_house_value"] for p in body["predictions"]]) - model_v1.predict(X_test[:200])).max()
        facts["max_diff"] = float(diff)
        return diff < 1e-4 and body["model_version"] == "housing-hgb-1.0.0", f"{body['n']} rows, max |container − offline| = {diff:.1e}"

    @step("9. load test", needs_docker=True)
    async def s9():
        payloads = [dict(zip(FEATURES, map(float, r))) for r in X_test[:300]]
        url = f"http://127.0.0.1:{facts['port']}/v1/predict"
        facts["latency"] = [await latency_test(url, payloads, 300, c, {"X-API-Key": API_KEY}) for c in (1, 8)]
        errors = sum(r["errors"] for r in facts["latency"])
        at8 = facts["latency"][1]
        return errors == 0, f"concurrency 8: p50 {at8['p50']:.1f} ms, p95 {at8['p95']:.1f} ms, p99 {at8['p99']:.1f} ms, {at8['rps']:.0f} req/s"

    @step("10. graceful stop", needs_docker=True)
    async def s10():
        stop = run([DOCKER, "stop", "-t", "15", release_container], show=False)      # grace period longer than any request
        code = run([DOCKER, "inspect", "-f", "{{.State.ExitCode}}", release_container], show=False).stdout.strip()
        graceful = "model unloaded" in run([DOCKER, "logs", release_container], show=False).stdout
        remove_container(release_container)
        return code == "0" and graceful, f"stopped in {stop.seconds:.1f} s, exit code {code}, shutdown hook ran: {graceful}"

    for name, needs_docker, needs_uv, fn in steps:
        if blocked():
            record(name, "not run", 0.0, "an earlier step failed")
            continue
        if needs_docker and not DOCKER_OK:
            record(name, "skipped", 0.0, "Docker daemon not available")
            continue
        if needs_uv and UV is None:
            record(name, "skipped", 0.0, "uv not installed")
            continue
        start = time.perf_counter()
        try:
            ok, detail = await fn()
        except Exception as err:                                  # noqa: BLE001 — a CI runner reports any failure
            ok, detail = False, f"{type(err).__name__}: {str(err)[:150]}"
        record(name, "passed" if ok else "failed", time.perf_counter() - start, detail)
    remove_container(release_container)


await run_pipeline()

### Step 3 — Release report (computed)

In [ ]:
passed = all(p["status"] in ("passed", "skipped") for p in pipeline)
skipped = [p["step"] for p in pipeline if p["status"] == "skipped"]
print(f"PIPELINE: {'PASSED' if passed else 'FAILED'} in {sum(p['seconds'] for p in pipeline):.0f} s"
      + (f" (skipped: {', '.join(skipped)})" if skipped else ""))

if DOCKER_OK and "size" in facts and "latency" in facts:
    naive_disk, naive_pull = sizes["housing-api:naive"]
    release_disk, release_pull = facts["size"]
    at1, at8 = facts["latency"]
    SLO_P95_MS = 50
    print(f"• Image: {release_disk / 1e6:.0f} MB on disk / {release_pull / 1e6:.0f} MB to pull — "
          f"{1 - release_pull / naive_pull:.0%} less to pull than the naive image, non-root, with a HEALTHCHECK.")
    print(f"• Cold start: ready {facts['cold_start']:.2f} s after `docker run`; Docker marked it healthy after {facts['healthy_after']:.1f} s.")
    print(f"• Correctness: host tests '{facts['host_tests']}' and in-image tests '{facts['image_tests']}'; "
          f"container predictions match the offline model (max diff {facts['max_diff']:.1e}).")
    print(f"• Latency: p95 {at1['p95']:.1f} ms alone, {at8['p95']:.1f} ms at concurrency 8 → SLO 'p95 < {SLO_P95_MS} ms at 8' "
          f"{'MET' if at8['p95'] < SLO_P95_MS else 'NOT MET'}; throughput {at1['rps']:.0f} → {at8['rps']:.0f} req/s "
          f"({at8['rps'] / at1['rps']:.1f}× with 8 concurrent clients on one container).")
    image_id = run([DOCKER, "image", "inspect", "-f", "{{.Id}}", RELEASE_TAG], show=False).stdout.strip()
    print(f"• Release: {RELEASE_TAG} = {image_id[:19]}… — deploy and roll back by this immutable ID (after a push, by its registry digest).")
elif not passed:
    first_failure = next(p for p in pipeline if p["status"] == "failed")
    print(f"Stopped at '{first_failure['step']}': {first_failure['detail']} — fix it and re-run, exactly as you would after a red CI build.")
else:
    print("Docker steps were skipped, so image size, cold start, and latency could not be measured on this machine.")

**Stretch goals**
1. Push the repo to GitHub and let the real workflow run. Then create a tag `v1.0.0` and watch the `publish` job push to GHCR.
2. Add a Trivy scan step to the `docker` job that fails on fixable HIGH/CRITICAL vulnerabilities.
3. Build a multi-architecture image (`docker buildx build --platform linux/amd64,linux/arm64`) so it runs on both x86 servers and Apple Silicon/Graviton.
4. Move the model out of the image: download a pinned version from object storage at startup, and compare image size and cold start.
5. Add a `deploy` job that starts a canary and runs `canary_decision()` from Exercise 6 on real metrics before promoting.

### 🗣️ How to talk about this in an interview
- "I containerized a FastAPI + scikit-learn service with a three-stage Dockerfile: a uv builder, a slim non-root runtime with a health check, and a test stage that runs the full test suite inside the exact runtime image."
- "Compared with a naive `FROM python:3.12` + `COPY . .` image, it was about 76% smaller to pull. I also showed that a careful single-stage slim image was similar in size here, because every wheel was pre-built. Multi-stage mainly buys isolation of build tools and a test target."
- "CI runs lint, a data-validation and model-quality gate that compares against production on a fixed holdout, host tests, tests inside the image, a container smoke test, and publishes versioned images only on tags, with least-privilege permissions and pinned actions."
- "I measured cold start and p50/p95/p99 latency on the container, checked predictions match the offline model, and verified graceful shutdown. Rollback means redeploying the previous immutable image ID or digest."
- "For rollout I'd use a canary with automated analysis: statistical comparison of error rates and model metrics, with 'wait' when there isn't enough traffic."

## 🎤 Interview Q&A

Try answering **out loud** before opening each answer.

### 🧠 Concepts

**Q1. What's the difference between a container and a virtual machine?**

<details><summary>Show answer</summary>

- **30-second answer:** A VM virtualizes hardware and runs its own OS kernel; a container is an isolated process sharing the host's kernel, fenced off with namespaces (what it can see) and cgroups (what it can use). Containers start in milliseconds and are MBs; VMs take much longer and are GBs, but isolate more strongly.
- **Go deeper:** Linux containers need a Linux kernel, which is why Docker Desktop runs a small VM on macOS and Windows (we saw `linuxkit` inside the container). Cloud Kubernetes nodes are VMs running containers. gVisor, Kata Containers, and Firecracker microVMs add VM-like isolation for untrusted code.
- **❌ Common wrong answer:** "A container is a lightweight VM with its own kernel."

</details>

**Q2. How does Docker layer caching work, and how should you order a Dockerfile?**

<details><summary>Show answer</summary>

- **30-second answer:** Each step's cache key is the parent layer, the instruction text, and (for `COPY`/`ADD`) a checksum of the copied files' contents and permissions. The first step whose key changes, and every step after it, is rebuilt. So order steps from rarely to frequently changing: base → system packages → copy dependency manifest → install → copy code.
- **Go deeper:** Modification times don't matter; ignored files don't matter (both verified in 🔧 Build It From Scratch). Use BuildKit cache mounts for package managers, `--cache-from`/`cache-to` (e.g. `type=gha`) to share cache in CI, and pin base images by digest for reproducible keys.
- **❌ Common wrong answer:** "`COPY . .` first is fine because Docker only rebuilds what changed."

</details>

**Q3. What is a multi-stage build and why use one?**

<details><summary>Show answer</summary>

- **30-second answer:** Several `FROM` stages in one Dockerfile. You build in a stage with compilers, toolchains, and caches, then `COPY --from=builder` only the results into a clean runtime stage. The final image excludes build tools, which cuts size and attack surface.
- **Go deeper:** Also useful for a `test` stage (run tests against the exact runtime image), and separate targets for dev and prod (`--target`). The size win depends on what the build needs: in this notebook, with pre-built wheels, a careful single-stage slim image was about the same size, while CUDA `-devel`→`-runtime` or compiled extensions save gigabytes.
- **❌ Common wrong answer:** "Multi-stage builds make the application run faster."

</details>

**Q4. Why shouldn't containers run as root?**

<details><summary>Show answer</summary>

- **30-second answer:** If the app is exploited, root inside the container can modify code and install tools (we overwrote the app code as root, and got "permission denied" as uid 10001), and has a far better chance of escaping via kernel or runtime bugs. Many clusters enforce `runAsNonRoot`.
- **Go deeper:** Create a fixed-UID system user, `USER app`, keep code owned by root and read-only, and add a read-only root filesystem, dropped Linux capabilities, `no-new-privileges`, and seccomp profiles at run time. Rootless Docker/Podman reduce risk further.
- **❌ Common wrong answer:** "Containers are isolated, so root inside can't do any harm."

</details>

**Q5. What's the difference between CI, continuous delivery, and continuous deployment?**

<details><summary>Show answer</summary>

- **30-second answer:** CI automatically builds and tests every change so the main branch stays working. Continuous delivery also produces a versioned, deployable artifact for every good change (a human triggers release). Continuous deployment releases every passing change to production automatically.
- **Go deeper:** For ML, CI adds data validation and model quality gates, and CD often includes staged rollouts (canary, shadow) and automated rollback. "Continuous training" (CT) retrains on new data as another pipeline, feeding the same gates.
- **❌ Common wrong answer:** "CI/CD is just a tool like Jenkins or GitHub Actions."

</details>

**Q6. Blue/green vs canary deployment — when would you choose each?**

<details><summary>Show answer</summary>

- **30-second answer:** Blue/green runs two full environments and switches all traffic at once: instant cutover and rollback, at 2× capacity, but every user hits the new version at the same moment. Canary sends a small share of real traffic to the new version and ramps up only while metrics hold: smaller blast radius, but it needs good metrics and time.
- **Go deeper:** For models, canaries compare error rates, latency, and model or business metrics with statistical tests (our canary compared MAE with a bootstrap interval, and Exercise 6 used a z-test). Shadow deployments score traffic without serving it, which is safest for risky models. Keep routing sticky per user for consistent experiences.
- **❌ Common wrong answer:** "They're the same thing — both run two versions."

</details>

### 💻 Coding

**Q7. Sketch a production Dockerfile for a FastAPI + scikit-learn service.**

<details><summary>Show answer</summary>

- **30-second answer:** Builder: `FROM python:3.12-slim AS builder`, copy uv, copy `requirements.txt` (pinned), `uv venv /opt/venv && uv pip install -r` with a cache mount. Runtime: `FROM python:3.12-slim`, `ENV PATH=/opt/venv/bin:$PATH PYTHONUNBUFFERED=1`, create a non-root user, `COPY --from=builder /opt/venv`, copy `app/` and `model/`, `USER app`, `EXPOSE 8000`, `HEALTHCHECK`, `CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]`.
- **Go deeper:** `.dockerignore`; config and secrets from env at run time; pin the base by digest; model baked in vs downloaded; one process per container, scaled by replicas; the scikit-learn version must match the one that pickled the model; build for the server's architecture (`--platform linux/amd64`).
- **❌ Common wrong answer:** `FROM python:latest`, `COPY . .`, `pip install -r requirements.txt` (unpinned), `CMD uvicorn main:app` — unpinned, cache-busting, root, shell form, localhost binding.

</details>

**Q8. Your PyTorch inference image is 6 GB. How do you shrink it?**

<details><summary>Show answer</summary>

- **30-second answer:** Measure first (`docker history`, `dive`). Then: CPU-only wheels if serving on CPU, or a CUDA `-runtime` base instead of `-devel`; multi-stage so compilers and caches stay behind; no pip cache; only runtime dependencies; `.dockerignore`; clean up within the same `RUN`; move weights out of the image if they dominate.
- **Go deeper:** The CUDA-enabled PyTorch runtime image alone is several GB compressed (we queried Docker Hub). Consider ONNX Runtime or TensorRT for smaller runtimes, distroless bases, and removing test and training extras. Trade-off: downloading weights at startup slows cold starts unless nodes cache them.
- **❌ Common wrong answer:** "Use Alpine" — musl breaks most scientific wheels, forcing slow source builds that often end up bigger.

</details>

**Q9. Write a CI step that blocks a model worse than production.**

<details><summary>Show answer</summary>

- **30-second answer:** A script that loads the candidate model and a fixed, versioned holdout, validates the data schema first, computes the metric, compares it with absolute thresholds and the production model's recorded metric on the same holdout (e.g. ≤2% regression), writes a JSON report, and exits 1 on failure. The CI job runs it before building or publishing images.
- **Go deeper:** Add slice metrics (regions, user segments), fairness checks, latency and model size budgets, and statistical significance for noisy metrics. Store the report as an artifact. Our gate caught a label-misalignment bug, a subtly under-trained model (only by the production comparison), and corrupted data.
- **❌ Common wrong answer:** "Unit tests pass, so the model is fine to deploy."

</details>

### 🐛 Debugging Scenarios

**Q10. The image works on your laptop, but on the server the container exits immediately with `exec format error`. What happened?**

<details><summary>Show answer</summary>

- **30-second answer:** A CPU architecture mismatch: the image was built on Apple Silicon (linux/arm64) and the server is x86-64 (linux/amd64). Build for the target platform (`docker buildx build --platform linux/amd64`), or publish a multi-arch image for both.
- **Go deeper:** Check with `docker image inspect -f '{{.Architecture}}'`. Emulated builds (QEMU) are slow; native runners or cross-compilation are faster. Other "works locally" causes: files excluded by `.dockerignore` (⚠️ Pitfall 4), binding `127.0.0.1` (Pitfall 2), missing env vars, and different library versions than the pickled model expects.
- **❌ Common wrong answer:** "Docker images run identically everywhere, so it must be a server bug."

</details>

**Q11. `docker stop` (or a Kubernetes rollout) takes a long time and containers exit with code 137, dropping in-flight requests. Why?**

<details><summary>Show answer</summary>

- **30-second answer:** The app never received SIGTERM, typically because of shell-form `CMD` (so `/bin/sh` is PID 1) or a wrapper script without `exec`. After the grace period the runtime sends SIGKILL, and 137 = 128 + 9. Use exec-form `CMD`, `exec` in entrypoint scripts, or a tiny init (`tini`, `docker run --init`).
- **Go deeper:** Make the app handle SIGTERM gracefully (uvicorn does: stop accepting, finish in-flight requests, run lifespan shutdown). Set `terminationGracePeriodSeconds` above your longest request, and fail readiness during shutdown so the load balancer drains first. Exit code 137 can also mean OOM-killed: check `docker inspect` `.State.OOMKilled`.
- **❌ Common wrong answer:** "Increase the stop timeout" — it only makes the wait longer; the signal still doesn't arrive.

</details>

**Q12. After deploying a new model, error rates are normal but a business metric drops 8%. What do you do?**

<details><summary>Show answer</summary>

- **30-second answer:** Roll back first: redeploy the previous immutable image digest or move the model registry alias back to the previous version, then confirm the metric recovers. Only then investigate: training/serving skew, data drift, a slice regression, feature pipeline changes.
- **Go deeper:** Make rollback one command and practise it. Keep the previous version warm during rollouts. Log model version with every prediction to attribute impact. Add the missed check (slice metric, shadow comparison, longer canary) to the pipeline. Avoid schema or feature-store changes that make the old model incompatible.
- **❌ Common wrong answer:** "Debug the new model in production first to understand the root cause."

</details>

### 🏗️ Design

**Q13. How would you test an ML system end to end?**

<details><summary>Show answer</summary>

- **30-second answer:** A pyramid: many fast unit tests (features, formatting, with stub models), data tests (schema, nulls, ranges, distributions), model tests (thresholds on a fixed holdout, invariance and directional checks, slices), API contract tests (validation → 422, auth, readiness, parity with offline predictions), and a few end-to-end tests on the built container. Then online: shadow or canary analysis and monitoring for drift.
- **Go deeper:** Run the test suite inside the image. Version the holdout and the gate thresholds. Test training reproducibility (fixed seeds, pinned environments). Google's *ML Test Score* rubric is a good checklist for production readiness.
- **❌ Common wrong answer:** "Check accuracy on the test set once; ML models can't be unit-tested."

</details>

**Q14. Where should models and secrets be stored in a containerized ML platform?**

<details><summary>Show answer</summary>

- **30-second answer:** Models go in a model registry or versioned object storage (MLflow Model Registry, S3/GCS paths with immutable versions, Hugging Face Hub), referenced by an immutable version or digest; small models can be baked into immutable images. Secrets go in a secret manager (Vault, AWS/GCP Secret Manager, Kubernetes Secrets with encryption), injected at run time; never in images, `ENV`/`ARG`, git, or logs.
- **Go deeper:** Record lineage with each model version (data version, code commit, metrics, library versions). CI authenticates to clouds via OIDC with short-lived credentials. Build-time secrets use BuildKit secret mounts. Rotate keys, and scan images and repos for leaked secrets.
- **❌ Common wrong answer:** "Put the API key in the Dockerfile as an `ENV` — the image is private anyway."

</details>

## 🧪 Quick Quiz

Predict the answer, then reveal. Each answer was verified by a real build or run in this notebook.

**1.** With the cache-friendly Dockerfile from Section 2 (`COPY requirements.txt` → `RUN pip install` → `COPY app.py`), you change one line of `app.py` and rebuild. Which steps actually run?
<details><summary>Answer</summary>

Only **`COPY app.py .`**: `WORKDIR`, `COPY requirements.txt`, and `RUN pip install` all showed `CACHED`.
</details>

**2.** You `touch` a source file (new modification time, same contents) and rebuild. Does its `COPY` step rebuild?
<details><summary>Answer</summary>

**No.** BuildKit's cache key hashes file contents and permissions, not modification times (🔧 Build It From Scratch, scenario 1). A `chmod +x`, however, *does* trigger a rebuild.
</details>

**3.** A Dockerfile creates a 60 MB file in one `RUN` and deletes it in the next `RUN`. How much does that add to the image?
<details><summary>Answer</summary>

**About 60 MB.** The file still lives in the earlier layer (⚠️ Pitfall 3). Deleting it in the same `RUN` adds almost nothing.
</details>

**4.** `yaml.safe_load` on a GitHub Actions workflow: what is the key for the `on:` section?
<details><summary>Answer</summary>

The boolean **`True`**. PyYAML follows YAML 1.1, where `on` means true (Section 11).
</details>

**5.** With `CMD uvicorn app.main:app --host 0.0.0.0` (no JSON brackets), what is PID 1 inside the container?
<details><summary>Answer</summary>

**`/bin/sh -c "uvicorn …"`**. The shell, not uvicorn, receives `docker stop`'s SIGTERM, so the app is SIGKILLed with exit code 137 and its shutdown code never runs (⚠️ Pitfall 1).
</details>

## 📚 Resources

### 📖 Official Docs
- [Dockerfile reference](https://docs.docker.com/reference/dockerfile/) — every instruction, exec vs shell form, `HEALTHCHECK` options
- [Docker build cache](https://docs.docker.com/build/cache/) — how cache invalidation works, cache mounts, external cache in CI
- [Multi-stage builds](https://docs.docker.com/build/building/multi-stage/) — the official guide to stages and `--target`
- [Build secrets](https://docs.docker.com/build/building/secrets/) — `RUN --mount=type=secret`, as used in Section 7
- [Using uv in Docker](https://docs.astral.sh/uv/guides/integration/docker/) — the uv team's recommended Dockerfile patterns
- [Compose file reference](https://docs.docker.com/reference/compose-file/) — services, `depends_on`, health conditions
- [Workflow syntax for GitHub Actions](https://docs.github.com/en/actions/reference/workflows-and-actions/workflow-syntax) — `on`, `jobs`, `needs`, `if`, `permissions`
- [Kubernetes — Liveness, Readiness and Startup Probes](https://kubernetes.io/docs/tasks/configure-pod-container/configure-liveness-readiness-startup-probes/) — how orchestrators use health endpoints

### 🎥 Videos
- [TechWorld with Nana — Docker Crash Course for Absolute Beginners](https://www.youtube.com/watch?v=pg19Z8LL06w) (~1 h) — images, containers, ports, and Dockerfiles explained visually; a good first pass before Sections 1–5
- [Fireship — 100+ Docker Concepts you Need to Know](https://www.youtube.com/watch?v=rIrNIzy6U_g) (~9 min) — a fast, dense review of the vocabulary interviewers use
- [TechWorld with Nana — GitHub Actions Tutorial - Basic Concepts and CI/CD Pipeline with Docker](https://www.youtube.com/watch?v=R8_veQiYBjI) (~33 min) — workflows, jobs, and building/pushing images, matching Section 11

### 📄 Papers
- [Breck et al. (2017) — The ML Test Score: A Rubric for ML Production Readiness and Technical Debt Reduction](https://research.google/pubs/the-ml-test-score-a-rubric-for-ml-production-readiness-and-technical-debt-reduction/) — Google's checklist of data, model, infrastructure, and monitoring tests (Sections 12–13)
- [Sculley et al. (2015) — Hidden Technical Debt in Machine Learning Systems](https://papers.nips.cc/paper_files/paper/2015/hash/86df7dcfd896fcaf2674f757a2463eba-Abstract.html) — why pipelines, configuration, and serving dominate ML system cost

### 📘 Books & Courses
- [Google Cloud — MLOps: Continuous delivery and automation pipelines in machine learning](https://cloud.google.com/architecture/mlops-continuous-delivery-and-automation-pipelines-in-machine-learning) — the widely cited MLOps maturity levels (CI, CD, and continuous training for ML)
- [Made With ML](https://madewithml.com/) — Goku Mohandas's free course on testing, CI/CD, and serving ML systems

### 🏋️ Practice
- [Play with Docker](https://labs.play-with-docker.com/) — a free browser sandbox to practise Docker commands without installing anything
- [GitHub Skills — Hello GitHub Actions](https://github.com/skills/hello-github-actions) — build your first workflow in a guided repository

## 📝 Summary Cheat Sheet

| Concept | What it does | Key command / rule |
|---|---|---|
| Container vs VM | isolated process on a shared kernel vs full virtual machine | namespaces + cgroups; Docker Desktop = one Linux VM |
| Image & layers | read-only stack, one layer per `RUN`/`COPY`/`ADD` | `docker history`, `docker image inspect` |
| Build cache | reuse a step if parent + instruction + copied contents match | order: base → deps manifest → install → code |
| Build context | files sent to the builder | `.dockerignore`; copy explicit paths |
| Run | start, publish ports, configure | `docker run -d -p 127.0.0.1:8080:8000 -e KEY=… image`; `logs`, `exec`, `stop`, `rm` |
| Multi-stage | build tools stay out of the runtime image | `FROM … AS builder` → `COPY --from=builder`; `--target` |
| Size | faster pulls, less attack surface | slim base, no caches, runtime deps only, same-`RUN` cleanup |
| Security | limit damage, never leak secrets | `USER app`; secrets at run time; `RUN --mount=type=secret` |
| Health & config | orchestration and env-specific settings | `HEALTHCHECK`; `ENV` defaults overridden with `-e` |
| Model artifacts | baked in vs mounted/downloaded | immutable versions; `-v host:/models:ro` + `MODEL_DIR` |
| Signals | graceful shutdown | exec-form `CMD ["uvicorn", …]`; 137 = SIGKILL |
| Compose | multi-container local/CI stacks | `docker compose up --exit-code-from tests`; `depends_on: service_healthy` |
| GPU images | CUDA libs in image, driver on host | `-runtime` not `-devel`; `--gpus all`; CPU wheels for CPU serving |
| GitHub Actions | CI/CD workflows | `on`, `jobs`, `needs`, `if`, pinned `uses`, least-privilege `permissions` |
| ML testing pyramid | unit → data → model → API → e2e | stub models, fixed holdout, invariance/directional tests |
| Quality gate | block worse models | data validation first; absolute + vs-production thresholds; exit 1 |
| Rollouts | ship safely, roll back fast | rolling · blue/green · canary · shadow; redeploy previous digest |

## ➡️ What's Next

**[03 · BentoML](03_BentoML.ipynb)** — you've hand-built the serving stack: FastAPI, Docker, and CI. BentoML packages the same ideas (model store, adaptive batching, generated containers) into one framework, so you can compare doing it yourself with a purpose-built ML serving tool.